# Exp29 (v5, run 4) — Direction-Transfer Test (DAS-lite): Gender vs Word Order

> **v5 changes are listed in the next cell.** The rest of this cell describes v4 and is kept for the record.

Fresh, self-contained follow-up to Exp27 (word-order whole-state patching → INVERSION_ABANDONED) and Exp28 (gender whole-state patching → INSTRUMENT_INADEQUATE). Question: is there a direction at the decision token that (a) moves the readout inside its source language, (b) is *necessary* there, and (c) moves the same readout in other languages — and is (c) larger for gender than for word order, *specifically*?

**What changed relative to whole-state patching.** Whole-state patching overwrites language identity, content and the feature at once, so a null can mean "readout is language-specific" and a positive can be non-specific disruption. Here we add/remove a single direction, and a cell only counts as TRANSFER if sufficiency **and** necessity pass in the source language and every control stays quiet (triangulation standard).

**Method, honestly labelled.**
- *DAS-lite, not full DAS.* Boguraev et al. (arXiv:2608.28924) learn a 1-D subspace by backprop. We use a gradient-free SPSA fit restricted to span{diff-of-means axis, top-k derivation-state PCs}, initialised at the diff-of-means direction. The fitted direction replaces diff-of-means **only if it wins on held-out derivation items**; the run prints how often that happened. If it rarely does, read the results as "diff-of-means with an SPSA sanity check", not as DAS.
- *Everything that chooses something uses the derivation half only* (window, directions, layer L*, alpha*). The test half only evaluates. Recipients are scored on held-out scenes/concepts.
- *Necessity* = knocking the direction out across the whole window (single-layer ablation gets undone by later layers) removes ≥ `necessity_min_drop` of the native margin **and** ≥ 3× what knocking out random directions removes. (This replaces the earlier "flip rate under knockout" wording, which would almost never fire: ablation neutralises, it rarely reverses.)
- *Controls:* 50 random directions, 200 label-permutation directions, a real language-ID cue direction (neutral sentences, both signs), the dass-vs-ob nuisance direction for order, and 2×2 specificity (own-task vs other-task direction, diff-of-means, same layer/alpha) applied to any cell that would otherwise be TRANSFER.
- *Hub check:* FR↔ES are both Romance, so Romance-pair transfer can reflect proximity rather than gender. AR/HI sources into FR/ES are reported separately; the verdict is built from the four primary cells only.

**Data status.** Both item banks are populated. FR/ES/DE are standard. HI/AR gender items were cross-checked against dictionaries/grammar references where I could, and HI/JA/KO/TR order sentences are simple present-tense SVO written by me — **none of HI/AR/JA/KO/TR has had a native-speaker pass.** The clean-accuracy gate catches grossly wrong items but cannot catch a fluent-but-wrong one; treat HI/AR cells as provisional in any write-up.

**Run history.** Run 1: calibration saturated (alpha* on the grid floor, fit loss flat). Run 2: grid fixed, but 10 gender concepts / 8-16 order units per half left every blocker at a one-item margin, and alpha* = 'smallest alpha reaching 0.5 on derivation' makes a 0.5 test gate fail ~half the time by regression to the mean. Run 3 (this file): 20 gender concepts per half, 12 order scenes per half, `alpha_target_flip = 0.65` (**rule-v2**, applied identically to gender and order), plus dose-response curves and a fitted-vs-diff-of-means comparison. Run 3 is a *tuning* run on Gemma-2-9B-it; freeze rule-v2 and confirm on a model not used for tuning before reporting a verdict.

**What this does not establish.** A transferring direction shows an additive, language-portable code exists at this readout — not where or how it was computed, nor that the same heads/MLPs produce it in each language (candidate Exp30). Write "transfers through this readout", not "shared computation".


## v5 (run 4) — what changed from v4, and why (all of it written AFTER run 3, BEFORE any confirmatory run)

Run 3 (v4 rules) returned INSTRUMENT_INADEQUATE. Reading its printout and code showed three problems with the *instrument*, none of which is a finding about gender or word order:

1. **Layer mismatch.** v4 injected every source's direction at the *source's* L\* in every recipient. AR's L\* is 24 while FR/ES/HI are 32-34, so AR-sourced cells (AR→HI WALL, AR→FR 0.44) and every order cell (DE only has L\*=28) were tested at a single layer that may not be where the recipient reads the feature. **v5:** inject the source's layer-*L* direction at layer *L*; choose *L* on the **recipient derivation half** among layers where the **source** direction reaches the 0.65 derivation target at α\* (the rule-v2 headroom, so the 0.5 test gate at that layer is not failed by regression to the mean); evaluate on the test half. R divides by the source's own in-language flip *at that same layer*. The v4-style number (inject at L\*) is still reported (`R_at_Lstar_v4style`) so run 4 can be compared with run 3 directly.
2. **The absolute control rule (every control ≤ 0.10) is noise-limited and bears on the wrong cells.** It killed ES→FR (cue 0.15 = 3/20 items), HI→FR (permutation mean 0.11, recipient flip 1.00, R = 1.06) and DE→FR (cue 0.25, a cell with *zero* own flip, where it cannot bear on any claim). **Rule-v3:** controls are reported for every cell but *gate only cells that claim transfer* (R ≥ 0.5): own recipient flip must be ≥ 3× max(random, permutation, cue, 1/n_items). The 2×2 specificity and the dass/ob nuisance checks are unchanged.
3. **Two unpowered claims.** (a) A WALL from 0 of 12 flips cannot exclude a true flip rate of ~0.25; bootstrap CIs of an all-zero sample are `[0,0]`, an artifact. **v5:** WALL needs the exact (Clopper-Pearson) 95% upper bound of R ≤ 0.5, else `WALL_LOWPOWER` (verdict: `INCONCLUSIVE_LOWPOWER`, never "wall"); `R_ub` is reported so the stronger "R < 0.3" claim can be read off. (b) R saturates at recipient flip = 1.0 (HI→FR and HI→ES have identical R and CI). **v5:** per-cell dose-response out to α = 4 (grid + [3, 4], used for curves only, **not** to pick α\*), a random-direction curve at every α, and `ED50` ratios (source ED50 / recipient ED50; 0.0 = recipient never reaches 0.5 anywhere on the grid).

**Unchanged:** item banks, gates, window/L\*/α\* selection, SPSA (still reported; call it diff-of-means), necessity gate, the four primary cells, the verdict tree.

**Still not done (Exp29b):** order directions that are not German-derived, and a within-language order contrast in a second family. Without one, language identity and order class are confounded by construction, so an order WALL here means "the German V2/verb-final frame direction does not move these recipients", not "order is not shared".

**Freeze rule.** `prereg.json` is written to `exp29_results_run4/` on first run. The 9B run is a tuning run; copy the notebook unchanged, point `RESULTS_DIR` to a new directory, and run Gemma-2-27B-it for the confirmatory result. Any further rule change is a new study.


In [1]:
import json, os

RESULTS_DIR = "exp29_results_run4"   # run 1 (exp29_results/) saturated; run 3 (exp29_results_run3/) = v4 rules, kept for the record. Run 4 = v5 (rule-v3). A changed rule is a NEW dir.
os.makedirs(RESULTS_DIR, exist_ok=True)

CONFIG = dict(
    # --- gates ---
    gate_acc                 = 0.75,   # per-language clean-accuracy gate (native class preferred over the other)
    gate_min_items            = 8,      # usable (tokenisation-clean) items needed in EACH of derivation / test halves
    min_correct_items         = 6,      # clean-correct items needed to compute any effect
    alpha_target_flip         = 0.65,   # RULE-V2: derivation-half flip target used to pick alpha* (run 1-2 used 0.50 = suff_min_flip)
    suff_min_flip             = 0.50,   # in-language SUFFICIENCY: own-direction flip rate at alpha*, test half
    necessity_min_drop        = 0.30,   # in-language NECESSITY: band-knockout removes >= this FRACTION of the native margin (test half)...
    necessity_vs_random_ratio = 3.0,    # ...and at least this many times what knocking out random directions removes
    # --- RULE-V3 (written after run 3, BEFORE any confirmatory run; supersedes the absolute control_max_flip = 0.10) ---
    control_ratio_min         = 3.0,    # a cell that CLAIMS transfer (R >= transfer_min) needs own recipient flip >= this x max(random, permutation, cue, floor). Controls are REPORTED for every cell but GATE only claimed-transfer cells.
    control_floor_items       = 1,      # floor in the ratio = this / n_recipient_items (one flipped item is noise, not a control effect)
    nuisance_floor            = 0.10,   # order nuisance (dass/ob) direction: allowed flip = max(this, 0.5 x own flip)
    wall_ub_max               = 0.5,    # WALL needs the exact 95% upper bound of R <= this (the cell must be able to EXCLUDE transfer); otherwise WALL_LOWPOWER
    # --- recipient layer rule (v5): inject the source's layer-L direction at layer L, L chosen on the RECIPIENT derivation half among layers where the SOURCE direction is itself sufficient (derivation half) ---
    rec_layer_metric          = "deriv_shift",
    # --- dose-response extension (recipient curves only; NOT used to pick alpha*) ---
    alpha_dose_extra          = [3, 4],
    n_dose_rand               = 8,
    rule_version              = "v3: recipient-layer sweep + ratio controls (claims only) + exact-CI walls",
    specificity_min_ratio     = 3.0,    # 2x2: own-task flip >= this x cross-task flip (only applied to cells that would be TRANSFER)
    wall_max                  = 0.20,   # R <= this counts as WALL
    transfer_min              = 0.50,   # R >= this counts as TRANSFER
    ci_separation_required    = True,   # INVERSION_SUPPORTED requires non-overlapping 95% CIs of R_gender vs R_order
    # --- DAS-lite: SPSA fit inside span{diff-of-means, top derivation-state PCs} ---
    das_spsa_steps            = 150,
    das_spsa_c                = 0.1,    # probe radius (radians) on the unit sphere of the subspace
    das_spsa_lr               = 1.0,    # first step = lr * c radians, decays ~1/sqrt(t)
    das_subspace_k            = 8,      # PCs added to the diff-of-means axis
    das_alpha_fit             = 0.15,   # steering strength while fitting. Run 1 used 2.0, which saturated (every item flipped -> flat loss -> SPSA never moved)
    das_holdout_frac          = 0.2,    # derivation items held out INSIDE the derivation half, early stopping only
    das_patience              = 40,
    das_min_improvement       = 0.05,   # fitted direction replaces diff-of-means only if held-out loss improves by >= this (nats)
    # --- alpha* sweep ---
    alpha_grid                = [0.02, 0.05, 0.1, 0.2, 0.35, 0.5, 1, 2],   # run 1 grid started at 0.5 and alpha* sat on the floor
    alpha_probe               = 0.25,    # strength used for the leave-one-out layer curve (window selection)
    # --- layer window (leave-one-out diff-of-means curve on the DERIVATION half only) ---
    window_frac_of_max        = 0.5,
    # --- controls ---
    n_random_dirs             = 50,
    n_perm_dirs               = 200,
    # --- scenes / bootstrap ---
    min_scenes                = 8,
    min_xfam_scenes           = 4,
    N_BOOT                    = 2000,
    LAYER_STRIDE              = 2,
    noise_floor_mult          = 1.5,
)

cfg_path = os.path.join(RESULTS_DIR, "prereg.json")
cfg_str = json.dumps(CONFIG, sort_keys=True, indent=2)
if os.path.exists(cfg_path):
    prev = open(cfg_path).read()
    if prev.strip() != cfg_str.strip():
        raise RuntimeError(
            "prereg.json already exists and differs from this run's CONFIG. "
            "A changed rule is a new study: use a new RESULTS_DIR, don't edit this one.\n"
            f"--- existing ---\n{prev}\n--- this run ---\n{cfg_str}"
        )
else:
    open(cfg_path, "w").write(cfg_str)

print("prereg frozen at", cfg_path)


prereg frozen at exp29_results_run4/prereg.json


In [2]:
import os

if 'model' not in globals() or model is None:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2")  # MUST be set before importing torch
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    MODEL_PATH = os.environ.get("EXP29_MODEL_PATH", "/home/models/gemma-2-9b-it")
    tok = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, torch_dtype=torch.bfloat16, attn_implementation="eager", device_map={"": 0}
    )
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)   # frozen: SPSA never needs gradients through the model
else:
    import torch  # already provided by the caller (e.g. the toy test harness)

print("model ready:", type(model).__name__)


/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 464/464 [00:02<00:00, 188.87it/s]


model ready: Gemma2ForCausalLM


In [3]:
import numpy as np

def get_layers(m):
    for attr_path in [("model", "layers"), ("transformer", "h"), ("gpt_neox", "layers")]:
        obj = m
        ok = True
        for a in attr_path:
            if hasattr(obj, a):
                obj = getattr(obj, a)
            else:
                ok = False
                break
        if ok:
            return obj
    raise AttributeError("could not locate decoder layers on model")

LAYERS = get_layers(model)
N_LAYERS = len(LAYERS)

def encode_unit(prefix_text, full_text, tokenizer):
    """Tokenize full_text; verify prefix_text's tokenization is an EXACT token-prefix of it.
    Returns (full_ids, decision_pos) or None if the unit fails the prefix-consistency check
    (dropped units are counted by the caller, never silently excluded)."""
    p_ids = tokenizer(prefix_text)["input_ids"]
    f_ids = tokenizer(full_text)["input_ids"]
    if len(p_ids) > len(f_ids) or f_ids[:len(p_ids)] != p_ids:
        return None
    return f_ids, len(p_ids) - 1

def collate(id_lists, pad_id):
    T = max(len(x) for x in id_lists)
    ids = np.full((len(id_lists), T), pad_id, dtype=np.int64)
    mask = np.zeros((len(id_lists), T), dtype=np.int64)
    for i, x in enumerate(id_lists):
        ids[i, :len(x)] = x
        mask[i, :len(x)] = 1
    device = next(model.parameters()).device if ('model' in globals() and model is not None and hasattr(model, "parameters")) else torch.device("cpu")
    return torch.tensor(ids, device=device), torch.tensor(mask, device=device)

HOOK_REGISTRY = []

def _clear_hooks():
    for h in HOOK_REGISTRY:
        h.remove()
    HOOK_REGISTRY.clear()

def make_intervention_hook(mode, pos_list, vec=None, alpha=0.0, sigma=1.0):
    """mode: 'add'      h[i] += alpha*sigma*unit(vec)   (sufficiency / steering)
              'knockout' h[i] -= (h[i].u)u                (necessity / ablation)
       pos_list: per-row decision position (so a batch can mix scene lengths)."""
    def fn(module, inp, out):
        is_tuple = isinstance(out, tuple)
        h = out[0] if is_tuple else out
        hc = h.clone()
        u = None
        if vec is not None:
            nv = np.linalg.norm(vec) + 1e-8
            u_f = torch.tensor(vec / nv, dtype=torch.float32, device=hc.device); u = u_f.to(hc.dtype)
        for i, p in enumerate(pos_list):
            row = hc[i, p, :]
            if mode == "add":
                row = row + float(alpha) * float(sigma) * u
            elif mode == "knockout":
                row = (row.float() - (row.float() @ u_f) * u_f).to(hc.dtype)
            else:
                raise ValueError(mode)
            hc[i, p, :] = row
        return (hc,) + out[1:] if is_tuple else hc
    return fn

def forward_with_intervention(ids, mask, layer_idx, pos_list, mode, vec=None, alpha=0.0, sigma=1.0):
    device = next(model.parameters()).device if hasattr(model, "parameters") else ids.device
    ids, mask = ids.to(device), mask.to(device)
    _clear_hooks()
    h = LAYERS[layer_idx].register_forward_hook(
        make_intervention_hook(mode, pos_list, vec=vec, alpha=alpha, sigma=sigma)
    )
    HOOK_REGISTRY.append(h)
    try:
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=mask, use_cache=False)
        return out.logits
    finally:
        _clear_hooks()

def clean_forward(ids, mask):
    device = next(model.parameters()).device if hasattr(model, "parameters") else ids.device
    ids, mask = ids.to(device), mask.to(device)
    _clear_hooks()
    with torch.no_grad():
        return model(input_ids=ids, attention_mask=mask, use_cache=False).logits

def capture_states(ids, mask, pos_list):
    device = next(model.parameters()).device if hasattr(model, "parameters") else ids.device
    ids, mask = ids.to(device), mask.to(device)
    """Hooks every layer; returns states[N, n_layers, d] at each row's own decision position."""
    captured = [None] * N_LAYERS
    def mk(li):
        def fn(mod, inp, out):
            captured[li] = (out[0] if isinstance(out, tuple) else out).detach().float().cpu()
        return fn
    _clear_hooks()
    for li, L in enumerate(LAYERS):
        HOOK_REGISTRY.append(L.register_forward_hook(mk(li)))
    try:
        with torch.no_grad():
            model(input_ids=ids, attention_mask=mask, use_cache=False)
    finally:
        _clear_hooks()
    N = ids.shape[0]
    d = captured[0].shape[-1]
    out = np.zeros((N, N_LAYERS, d))
    for li in range(N_LAYERS):
        h = captured[li].numpy()
        for i in range(N):
            out[i, li, :] = h[i, pos_list[i], :]
    return out

def residual_norm(states_2d):
    # states_2d: [N, d] -> mean L2 norm (sigma_L for a given layer)
    return float(np.sqrt((states_2d ** 2).sum(-1)).mean())

def score_candidates(logits, cand_id_lists, pos_list):
    """Teacher-forced sum log-prob of each row's candidate continuation, starting at pos+1."""
    lg = logits.detach().float().cpu().numpy() if hasattr(logits, "numpy") else logits
    out = np.zeros(len(cand_id_lists))
    for i, cand in enumerate(cand_id_lists):
        p = pos_list[i]
        row = lg[i] - lg[i].max(-1, keepdims=True)
        logp = row - np.log(np.exp(row).sum(-1, keepdims=True))
        s, cur = 0.0, p
        for tid in cand:
            s += logp[cur, tid]
            cur += 1
        out[i] = s
    return out

def margin(logits, posA_ids, posB_ids, pos_list):
    """S = logP(class A continuation) - logP(class B continuation), per row."""
    return score_candidates(logits, posA_ids, pos_list) - score_candidates(logits, posB_ids, pos_list)

def hidden_dim():
    """Robust hidden-size lookup: real HF config first, toy-harness shim shape as fallback."""
    if hasattr(model, "config") and hasattr(model.config, "hidden_size"):
        return model.config.hidden_size
    return LAYERS[0].W1.shape[0]  # toy-harness Layer exposes W1: (D, 2D)

print(f"infra ready: {N_LAYERS} layers, hidden_dim={hidden_dim()}")


infra ready: 42 layers, hidden_dim=3584


In [4]:
# ---------------------------------------------------------------------------
# GENDER item bank: schema + validator + a 40-concept x 4-language bank.
#
# CONFIDENCE LEVELS (read before trusting any number that comes out of this
# notebook -- this is the single most important caveat in the whole file):
#   FR / ES  -- standard, high-confidence dictionary gender for common nouns.
#               Low risk.
#   HI / AR  -- NOT reviewed by a native speaker. Two tiers:
#     (a) web-cross-checked this session against real grammar references
#         (table, house, window, car, sun, book, tooth) -- one of these
#         (Arabic "tooth") was WRONG on first pass (I had it feminine; it's
#         masculine, confirmed via arabic.fi and madinaharabic) and is fixed
#         below. That correction is itself evidence this tier still needs a
#         human pass, not proof the rest is safe.
#     (b) built from well-documented morphological patterns (Arabic: ends in
#         ta marbuta -> feminine, else masculine-by-default, checked against
#         the known short list of irregular feminine-without-ta-marbuta
#         exceptions; Hindi: -aa/-ii ending heuristic plus common-word
#         recall) but NOT individually cross-checked against a source this
#         session.
#   Every HI/AR row below is tagged verified=False regardless of tier --
#   that flag means "native-speaker signed off", which hasn't happened.
#   Get a native-speaker pass on at least tier (b) before trusting a
#   published claim built on AR/HI cells. The clean-accuracy gate below is
#   an empirical check on the MODEL's behavior, not a check on whether
#   these translations are actually correct -- a wrong-but-systematic
#   translation could still pass the gate.
# ---------------------------------------------------------------------------
# Unit schema (one row per concept x language):
#   concept   : str, concept id (same id across languages -- not required to
#               share gender; used only for the xfam flag and bookkeeping)
#   lang      : 'FR' | 'ES' | 'HI' | 'AR'
#   prefix    : str, text up to (not including) the agreeing adjective
#   fem       : str, feminine adjective form + fixed terminator
#   masc      : str, masculine adjective form + fixed terminator
#   native    : 'fem' | 'masc' -- this concept's actual gender in this language
#   xfam      : bool -- True if this concept's gender DIFFERS between FR and ES
#   split     : 'derivation' | 'test'
#   verified  : bool -- native-speaker sign-off. True for FR/ES. False for
#               every HI/AR row (see confidence note above).

GENDER_TERM = {"FR": ".", "ES": ".", "HI": " है.", "AR": "."}
GENDER_ADJ = {
    "FR": {"fem": "belle", "masc": "beau"},
    "ES": {"fem": "bella", "masc": "bello"},
    "HI": {"fem": "अच्छी", "masc": "अच्छा"},
    "AR": {"fem": "كبيرة", "masc": "كبير"},
}

# (concept, split, {lang: (prefix, native_gender, verified)})
# Hindi prefix pattern: "यह <noun> बहुत" ("this <noun> very") + adjective + " है."
# Arabic prefix pattern: definite noun ("al-" + noun) + adjective + "."
_GENDER_TABLE = [
    ("river",    "derivation", {"FR": ("La rivière est", "fem", True),
                                 "ES": ("El río es", "masc", True),
                                 "HI": ("यह नदी बहुत", "fem", False),
                                 "AR": ("النهر", "masc", False)}),
    ("wall",     "derivation", {"FR": ("Le mur est", "masc", True),
                                 "ES": ("La pared es", "fem", True),
                                 "HI": ("यह दीवार बहुत", "fem", False),
                                 "AR": ("الجدار", "masc", False)}),
    ("bag",      "derivation", {"FR": ("Le sac est", "masc", True),
                                 "ES": ("La bolsa es", "fem", True),
                                 "HI": ("यह झोला बहुत", "masc", False),
                                 "AR": ("الحقيبة", "fem", False)}),
    ("plate",    "derivation", {"FR": ("L'assiette est", "fem", True),
                                 "ES": ("El plato es", "masc", True),
                                 "HI": ("यह थाली बहुत", "fem", False),
                                 "AR": ("الصحن", "masc", False)}),
    ("book",     "derivation", {"FR": ("Le livre est", "masc", True),
                                 "ES": ("El libro es", "masc", True),
                                 "HI": ("यह किताब बहुत", "fem", False),   # web-checked: kitab is the standard fem word for book
                                 "AR": ("الكتاب", "masc", False)}),       # web-checked: kitab = masculine (arabic.fi, madinaharabic)
    ("table",    "derivation", {"FR": ("La table est", "fem", True),
                                 "ES": ("La mesa es", "fem", True),
                                 "HI": ("यह मेज़ बहुत", "fem", False),     # web-checked: mez = unmarked feminine (hindipod101)
                                 "AR": ("الطاولة", "fem", False)}),
    ("door",     "derivation", {"FR": ("La porte est", "fem", True),
                                 "ES": ("La puerta es", "fem", True),
                                 "HI": ("यह दरवाज़ा बहुत", "masc", False),
                                 "AR": ("الباب", "masc", False)}),
    ("window",   "derivation", {"FR": ("La fenêtre est", "fem", True),
                                 "ES": ("La ventana es", "fem", True),
                                 "HI": ("यह खिड़की बहुत", "fem", False),   # web-checked: khirki = feminine (hindipod101)
                                 "AR": ("النافذة", "fem", False)}),
    ("house",    "derivation", {"FR": ("La maison est", "fem", True),
                                 "ES": ("La casa es", "fem", True),
                                 "HI": ("यह घर बहुत", "masc", False),      # web-checked: ghar = unmarked masculine (hindipod101)
                                 "AR": ("البيت", "masc", False)}),        # web-checked: bayt = masculine (note: "daar" would be an irregular-fem exception; bayt avoids that trap)
    ("sun",      "derivation", {"FR": ("Le soleil est", "masc", True),
                                 "ES": ("El sol es", "masc", True),
                                 "HI": ("यह सूरज बहुत", "masc", False),    # web-checked: suraj = masculine (lingq)
                                 "AR": ("الشمس", "fem", False)}),         # web-checked: shams = irregular feminine (arabic-for-nerds, kalimah-center, multiple sources)
    ("ring",     "test",       {"FR": ("La bague est", "fem", True),
                                 "ES": ("El anillo es", "masc", True),
                                 "HI": ("यह अंगूठी बहुत", "fem", False),
                                 "AR": ("الخاتم", "masc", False)}),
    ("tooth",    "test",       {"FR": ("La dent est", "fem", True),
                                 "ES": ("El diente es", "masc", True),
                                 "HI": ("यह दांत बहुत", "masc", False),
                                 "AR": ("السن", "masc", False)}),         # CORRECTED: web-checked masculine (arabic.fi, madinaharabic) -- first draft had this wrong as feminine
    ("car",      "test",       {"FR": ("La voiture est", "fem", True),
                                 "ES": ("El coche es", "masc", True),
                                 "HI": ("यह गाड़ी बहुत", "fem", False),    # web-checked: gaadi = feminine (lingq)
                                 "AR": ("السيارة", "fem", False)}),
    ("forest",   "test",       {"FR": ("La forêt est", "fem", True),
                                 "ES": ("El bosque es", "masc", True),
                                 "HI": ("यह जंगल बहुत", "masc", False),
                                 "AR": ("الغابة", "fem", False)}),
    ("room",     "test",       {"FR": ("La chambre est", "fem", True),
                                 "ES": ("La habitación es", "fem", True),
                                 "HI": ("यह कमरा बहुत", "masc", False),
                                 "AR": ("الغرفة", "fem", False)}),
    ("road",     "test",       {"FR": ("La rue est", "fem", True),
                                 "ES": ("La calle es", "fem", True),
                                 "HI": ("यह सड़क बहुत", "fem", False),
                                 "AR": ("الشارع", "masc", False)}),
    ("moon",     "test",       {"FR": ("La lune est", "fem", True),
                                 "ES": ("La luna es", "fem", True),
                                 "HI": ("यह चांद बहुत", "masc", False),
                                 "AR": ("القمر", "masc", False)}),        # web-checked: qamar = masculine (englishtoarabic.org, explicitly contrasted with shams)
    ("bridge",   "test",       {"FR": ("Le pont est", "masc", True),
                                 "ES": ("El puente es", "masc", True),
                                 "HI": ("यह पुल बहुत", "masc", False),
                                 "AR": ("الجسر", "masc", False)}),
    ("lake",     "test",       {"FR": ("Le lac est", "masc", True),
                                 "ES": ("El lago es", "masc", True),
                                 "HI": ("यह झील बहुत", "fem", False),
                                 "AR": ("البحيرة", "fem", False)}),
    ("mountain", "test",       {"FR": ("La montagne est", "fem", True),
                                 "ES": ("La montaña es", "fem", True),
                                 "HI": ("यह पहाड़ बहुत", "masc", False),
                                 "AR": ("الجبل", "masc", False)}),
    # ---- run-3 expansion: +10 derivation, +10 test concepts (HI/AR rows, as before, NOT natively reviewed) ----
    ("city",     "derivation", {"FR": ("La ville est", "fem", True), "ES": ("La ciudad es", "fem", True),
                                 "HI": ("यह शहर बहुत", "masc", False), "AR": ("المدينة", "fem", False)}),
    ("garden",   "derivation", {"FR": ("Le jardin est", "masc", True), "ES": ("El jardín es", "masc", True),
                                 "HI": ("यह बगीचा बहुत", "masc", False), "AR": ("الحديقة", "fem", False)}),
    ("school",   "derivation", {"FR": ("L'école est", "fem", True), "ES": ("La escuela es", "fem", True),
                                 "HI": ("यह स्कूल बहुत", "masc", False), "AR": ("المدرسة", "fem", False)}),
    ("pen",      "derivation", {"FR": ("Le stylo est", "masc", True), "ES": ("El bolígrafo es", "masc", True),
                                 "HI": ("यह कलम बहुत", "fem", False), "AR": ("القلم", "masc", False)}),
    ("chair",    "derivation", {"FR": ("La chaise est", "fem", True), "ES": ("La silla es", "fem", True),
                                 "HI": ("यह कुर्सी बहुत", "fem", False), "AR": ("الكرسي", "masc", False)}),
    ("bed",      "derivation", {"FR": ("Le lit est", "masc", True), "ES": ("La cama es", "fem", True),
                                 "HI": ("यह बिस्तर बहुत", "masc", False), "AR": ("السرير", "masc", False)}),
    ("bread",    "derivation", {"FR": ("Le pain est", "masc", True), "ES": ("El pan es", "masc", True),
                                 "HI": ("यह रोटी बहुत", "fem", False), "AR": ("الخبز", "masc", False)}),
    ("milk",     "derivation", {"FR": ("Le lait est", "masc", True), "ES": ("La leche es", "fem", True),
                                 "HI": ("यह दूध बहुत", "masc", False), "AR": ("الحليب", "masc", False)}),
    ("flower",   "derivation", {"FR": ("La fleur est", "fem", True), "ES": ("La flor es", "fem", True),
                                 "HI": ("यह फूल बहुत", "masc", False), "AR": ("الزهرة", "fem", False)}),
    ("tree",     "derivation", {"FR": ("L'arbre est", "masc", True), "ES": ("El árbol es", "masc", True),
                                 "HI": ("यह पेड़ बहुत", "masc", False), "AR": ("الشجرة", "fem", False)}),
    ("cloud",    "test",       {"FR": ("Le nuage est", "masc", True), "ES": ("La nube es", "fem", True),
                                 "HI": ("यह बादल बहुत", "masc", False), "AR": ("السحابة", "fem", False)}),
    ("sea",      "test",       {"FR": ("La mer est", "fem", True), "ES": ("El mar es", "masc", True),
                                 "HI": ("यह समुद्र बहुत", "masc", False), "AR": ("البحر", "masc", False)}),
    ("island",   "test",       {"FR": ("L'île est", "fem", True), "ES": ("La isla es", "fem", True),
                                 "HI": ("यह द्वीप बहुत", "masc", False), "AR": ("الجزيرة", "fem", False)}),
    ("clock",    "test",       {"FR": ("La montre est", "fem", True), "ES": ("El reloj es", "masc", True),
                                 "HI": ("यह घड़ी बहुत", "fem", False), "AR": ("الساعة", "fem", False)}),
    ("shirt",    "test",       {"FR": ("La chemise est", "fem", True), "ES": ("La camisa es", "fem", True),
                                 "HI": ("यह कमीज़ बहुत", "fem", False), "AR": ("القميص", "masc", False)}),
    ("shoe",     "test",       {"FR": ("La chaussure est", "fem", True), "ES": ("El zapato es", "masc", True),
                                 "HI": ("यह जूता बहुत", "masc", False), "AR": ("الحذاء", "masc", False)}),
    ("village",  "test",       {"FR": ("Le village est", "masc", True), "ES": ("El pueblo es", "masc", True),
                                 "HI": ("यह गाँव बहुत", "masc", False), "AR": ("القرية", "fem", False)}),
    ("bus",      "test",       {"FR": ("Le bus est", "masc", True), "ES": ("El autobús es", "masc", True),
                                 "HI": ("यह बस बहुत", "fem", False), "AR": ("الحافلة", "fem", False)}),
    ("tea",      "test",       {"FR": ("Le thé est", "masc", True), "ES": ("El té es", "masc", True),
                                 "HI": ("यह चाय बहुत", "fem", False), "AR": ("الشاي", "masc", False)}),
    ("key",      "test",       {"FR": ("La clé est", "fem", True), "ES": ("La llave es", "fem", True),
                                 "HI": ("यह चाबी बहुत", "fem", False), "AR": ("المفتاح", "masc", False)}),
]

def _xfam(entry):
    fr = entry["FR"][1]; es = entry["ES"][1]
    return fr != es

GENDER_UNITS = []
for concept, split, entry in _GENDER_TABLE:
    xfam = _xfam(entry)
    for lang, (prefix, native, verified) in entry.items():
        adj = GENDER_ADJ[lang]
        GENDER_UNITS.append(dict(
            concept=concept, lang=lang, prefix=prefix,
            fem=adj["fem"] + GENDER_TERM[lang], masc=adj["masc"] + GENDER_TERM[lang],
            native=native, xfam=xfam, split=split, verified=verified,
        ))

REQUIRED_PER_LANG_PER_SPLIT = 8   # derivation and test each, per language
MIN_PER_CLASS_PER_SPLIT = 2       # both fem and masc must appear at least this often

def validate_gender_bank(units, langs=("FR", "ES", "HI", "AR")):
    by = {}
    for u in units:
        by.setdefault((u["lang"], u["split"]), []).append(u)
    missing = []
    for lang in langs:
        for split in ("derivation", "test"):
            rows = by.get((lang, split), [])
            n = len(rows)
            if n < REQUIRED_PER_LANG_PER_SPLIT:
                missing.append(f"{lang}/{split}: have {n}, need >= {REQUIRED_PER_LANG_PER_SPLIT}")
                continue
            n_fem = sum(1 for r in rows if r["native"] == "fem")
            n_masc = n - n_fem
            if n_fem < MIN_PER_CLASS_PER_SPLIT or n_masc < MIN_PER_CLASS_PER_SPLIT:
                missing.append(f"{lang}/{split}: class imbalance (fem={n_fem}, masc={n_masc}), "
                                f"need >= {MIN_PER_CLASS_PER_SPLIT} of each")
    n_xfam_deriv = sum(1 for u in units if u["xfam"] and u["split"] == "derivation" and u["lang"] == "FR")
    n_xfam_test  = sum(1 for u in units if u["xfam"] and u["split"] == "test" and u["lang"] == "FR")
    if n_xfam_deriv < CONFIG["min_xfam_scenes"]:
        missing.append(f"xfam/derivation: have {n_xfam_deriv}, need >= {CONFIG['min_xfam_scenes']}")
    if n_xfam_test < CONFIG["min_xfam_scenes"]:
        missing.append(f"xfam/test: have {n_xfam_test}, need >= {CONFIG['min_xfam_scenes']}")
    return missing

_missing = validate_gender_bank(GENDER_UNITS)
GENDER_BANK_READY = (len(_missing) == 0)
n_unverified = sum(1 for u in GENDER_UNITS if not u["verified"])
if not GENDER_BANK_READY:
    print("GENDER item bank FAILS the count/balance check:")
    for m in _missing:
        print("  -", m)
else:
    print(f"GENDER item bank passes the count/balance check: {len(GENDER_UNITS)} rows, "
          f"{n_unverified} not natively verified (all HI/AR rows -- see confidence note above).")
    print("The clean-accuracy gate (Cell 6+) is the next real check; a native-speaker pass")
    print("on the HI/AR rows is still owed before trusting a published claim.")


GENDER item bank passes the count/balance check: 160 rows, 80 not natively verified (all HI/AR rows -- see confidence note above).
The clean-accuracy gate (Cell 6+) is the next real check; a native-speaker pass
on the HI/AR rows is still owed before trusting a published claim.


In [5]:
# ---------------------------------------------------------------------------
# ORDER item bank, authored fresh (not copied from Exp27): 24 simple transitive scenes
# x 8 languages. Scenes 0-11 = derivation half, 12-23 = test half.
#
# Each entry: (subject prefix, verb-object continuation, object-verb continuation).
# The decision token is the last token of the subject prefix. Readout:
#     S = logP(head-first / VO string) - logP(head-last / OV string)
#
# Native order: EN/FR/ES/DE head-first (VO); HI/JA/KO/TR head-last (OV).
# Deliberately inanimate objects (avoids Spanish personal 'a', Turkish/Hindi differential
# object marking quirks). Hindi verbs agree with subject gender (scenes 1, 3, 12, 14 are feminine).
#
# CONFIDENCE: same two-tier story as the gender bank. Simple present-tense SVO sentences with
# very common words, but NOT reviewed by native speakers (HI/JA/KO/TR especially). The
# clean-accuracy gate catches grossly wrong items; it cannot catch a fluent-but-wrong one.
# ---------------------------------------------------------------------------
ORDER_SCENES = [
 {"EN": ("The man", "reads the book.", "the book reads."), "FR": ("L'homme", "lit le livre.", "le livre lit."),
  "ES": ("El hombre", "lee el libro.", "el libro lee."), "DE": ("Der Mann", "liest das Buch.", "das Buch liest."),
  "HI": ("आदमी", "पढ़ता है किताब।", "किताब पढ़ता है।"), "JA": ("男の人は", "読む本を。", "本を読む。"),
  "KO": ("남자는", "읽는다 책을.", "책을 읽는다."), "TR": ("Adam", "okuyor kitabı.", "kitabı okuyor.")},
 {"EN": ("The woman", "opens the door.", "the door opens."), "FR": ("La femme", "ouvre la porte.", "la porte ouvre."),
  "ES": ("La mujer", "abre la puerta.", "la puerta abre."), "DE": ("Die Frau", "öffnet die Tür.", "die Tür öffnet."),
  "HI": ("औरत", "खोलती है दरवाज़ा।", "दरवाज़ा खोलती है।"), "JA": ("女の人は", "開けるドアを。", "ドアを開ける。"),
  "KO": ("여자는", "연다 문을.", "문을 연다."), "TR": ("Kadın", "açıyor kapıyı.", "kapıyı açıyor.")},
 {"EN": ("The boy", "eats the apple.", "the apple eats."), "FR": ("Le garçon", "mange la pomme.", "la pomme mange."),
  "ES": ("El chico", "come la manzana.", "la manzana come."), "DE": ("Der Junge", "isst den Apfel.", "den Apfel isst."),
  "HI": ("लड़का", "खाता है सेब।", "सेब खाता है।"), "JA": ("男の子は", "食べるりんごを。", "りんごを食べる。"),
  "KO": ("소년은", "먹는다 사과를.", "사과를 먹는다."), "TR": ("Oğlan", "yiyor elmayı.", "elmayı yiyor.")},
 {"EN": ("The girl", "drinks the water.", "the water drinks."), "FR": ("La fille", "boit l'eau.", "l'eau boit."),
  "ES": ("La niña", "bebe el agua.", "el agua bebe."), "DE": ("Das Mädchen", "trinkt das Wasser.", "das Wasser trinkt."),
  "HI": ("लड़की", "पीती है पानी।", "पानी पीती है।"), "JA": ("女の子は", "飲む水を。", "水を飲む。"),
  "KO": ("소녀는", "마신다 물을.", "물을 마신다."), "TR": ("Kız", "içiyor suyu.", "suyu içiyor.")},
 {"EN": ("The teacher", "writes the letter.", "the letter writes."), "FR": ("Le professeur", "écrit la lettre.", "la lettre écrit."),
  "ES": ("El profesor", "escribe la carta.", "la carta escribe."), "DE": ("Der Lehrer", "schreibt den Brief.", "den Brief schreibt."),
  "HI": ("अध्यापक", "लिखता है पत्र।", "पत्र लिखता है।"), "JA": ("先生は", "書く手紙を。", "手紙を書く。"),
  "KO": ("선생님은", "쓴다 편지를.", "편지를 쓴다."), "TR": ("Öğretmen", "yazıyor mektubu.", "mektubu yazıyor.")},
 {"EN": ("The child", "carries the bag.", "the bag carries."), "FR": ("L'enfant", "porte le sac.", "le sac porte."),
  "ES": ("El niño", "lleva la bolsa.", "la bolsa lleva."), "DE": ("Das Kind", "trägt die Tasche.", "die Tasche trägt."),
  "HI": ("बच्चा", "उठाता है बैग।", "बैग उठाता है।"), "JA": ("子供は", "持つかばんを。", "かばんを持つ。"),
  "KO": ("아이는", "든다 가방을.", "가방을 든다."), "TR": ("Çocuk", "taşıyor çantayı.", "çantayı taşıyor.")},
 {"EN": ("The cook", "makes the soup.", "the soup makes."), "FR": ("Le cuisinier", "prépare la soupe.", "la soupe prépare."),
  "ES": ("El cocinero", "prepara la sopa.", "la sopa prepara."), "DE": ("Der Koch", "kocht die Suppe.", "die Suppe kocht."),
  "HI": ("रसोइया", "बनाता है सूप।", "सूप बनाता है।"), "JA": ("料理人は", "作るスープを。", "スープを作る。"),
  "KO": ("요리사는", "만든다 수프를.", "수프를 만든다."), "TR": ("Aşçı", "yapıyor çorbayı.", "çorbayı yapıyor.")},
 {"EN": ("The driver", "washes the car.", "the car washes."), "FR": ("Le chauffeur", "lave la voiture.", "la voiture lave."),
  "ES": ("El conductor", "lava el coche.", "el coche lava."), "DE": ("Der Fahrer", "wäscht das Auto.", "das Auto wäscht."),
  "HI": ("ड्राइवर", "धोता है गाड़ी।", "गाड़ी धोता है।"), "JA": ("運転手は", "洗う車を。", "車を洗う。"),
  "KO": ("운전사는", "닦는다 차를.", "차를 닦는다."), "TR": ("Şoför", "yıkıyor arabayı.", "arabayı yıkıyor.")},
 {"EN": ("The doctor", "reads the report.", "the report reads."), "FR": ("Le médecin", "lit le rapport.", "le rapport lit."),
  "ES": ("El médico", "lee el informe.", "el informe lee."), "DE": ("Der Arzt", "liest den Bericht.", "den Bericht liest."),
  "HI": ("डॉक्टर", "पढ़ता है रिपोर्ट।", "रिपोर्ट पढ़ता है।"), "JA": ("医者は", "読む報告書を。", "報告書を読む。"),
  "KO": ("의사는", "읽는다 보고서를.", "보고서를 읽는다."), "TR": ("Doktor", "okuyor raporu.", "raporu okuyor.")},
 {"EN": ("The farmer", "buys the bread.", "the bread buys."), "FR": ("Le fermier", "achète le pain.", "le pain achète."),
  "ES": ("El granjero", "compra el pan.", "el pan compra."), "DE": ("Der Bauer", "kauft das Brot.", "das Brot kauft."),
  "HI": ("किसान", "ख़रीदता है रोटी।", "रोटी ख़रीदता है।"), "JA": ("農夫は", "買うパンを。", "パンを買う。"),
  "KO": ("농부는", "산다 빵을.", "빵을 산다."), "TR": ("Çiftçi", "alıyor ekmeği.", "ekmeği alıyor.")},
 {"EN": ("The student", "does the homework.", "the homework does."), "FR": ("L'étudiant", "fait le devoir.", "le devoir fait."),
  "ES": ("El estudiante", "hace la tarea.", "la tarea hace."), "DE": ("Der Student", "macht die Hausaufgabe.", "die Hausaufgabe macht."),
  "HI": ("छात्र", "करता है होमवर्क।", "होमवर्क करता है।"), "JA": ("学生は", "する宿題を。", "宿題をする。"),
  "KO": ("학생은", "한다 숙제를.", "숙제를 한다."), "TR": ("Öğrenci", "yapıyor ödevi.", "ödevi yapıyor.")},
 {"EN": ("The baker", "bakes the cake.", "the cake bakes."), "FR": ("Le boulanger", "cuit le gâteau.", "le gâteau cuit."),
  "ES": ("El panadero", "hornea el pastel.", "el pastel hornea."), "DE": ("Der Bäcker", "backt den Kuchen.", "den Kuchen backt."),
  "HI": ("नानबाई", "पकाता है केक।", "केक पकाता है।"), "JA": ("パン屋は", "焼くケーキを。", "ケーキを焼く。"),
  "KO": ("제빵사는", "굽는다 케이크를.", "케이크를 굽는다."), "TR": ("Fırıncı", "pişiriyor pastayı.", "pastayı pişiriyor.")},
 {"EN": ("The singer", "sings the song.", "the song sings."), "FR": ("La chanteuse", "chante la chanson.", "la chanson chante."),
  "ES": ("La cantante", "canta la canción.", "la canción canta."), "DE": ("Die Sängerin", "singt das Lied.", "das Lied singt."),
  "HI": ("गायिका", "गाती है गाना।", "गाना गाती है।"), "JA": ("歌手は", "歌う歌を。", "歌を歌う。"),
  "KO": ("가수는", "부른다 노래를.", "노래를 부른다."), "TR": ("Şarkıcı", "söylüyor şarkıyı.", "şarkıyı söylüyor.")},
 {"EN": ("The father", "drinks the tea.", "the tea drinks."), "FR": ("Le père", "boit le thé.", "le thé boit."),
  "ES": ("El padre", "bebe el té.", "el té bebe."), "DE": ("Der Vater", "trinkt den Tee.", "den Tee trinkt."),
  "HI": ("पिता", "पीता है चाय।", "चाय पीता है।"), "JA": ("お父さんは", "飲むお茶を。", "お茶を飲む。"),
  "KO": ("아버지는", "마신다 차를.", "차를 마신다."), "TR": ("Baba", "içiyor çayı.", "çayı içiyor.")},
 {"EN": ("The mother", "cooks the rice.", "the rice cooks."), "FR": ("La mère", "cuit le riz.", "le riz cuit."),
  "ES": ("La madre", "cocina el arroz.", "el arroz cocina."), "DE": ("Die Mutter", "kocht den Reis.", "den Reis kocht."),
  "HI": ("माँ", "पकाती है चावल।", "चावल पकाती है।"), "JA": ("お母さんは", "炊くご飯を。", "ご飯を炊く。"),
  "KO": ("어머니는", "짓는다 밥을.", "밥을 짓는다."), "TR": ("Anne", "pişiriyor pilavı.", "pilavı pişiriyor.")},
 {"EN": ("The player", "hits the ball.", "the ball hits."), "FR": ("Le joueur", "frappe le ballon.", "le ballon frappe."),
  "ES": ("El jugador", "golpea la pelota.", "la pelota golpea."), "DE": ("Der Spieler", "schlägt den Ball.", "den Ball schlägt."),
  "HI": ("खिलाड़ी", "मारता है गेंद।", "गेंद मारता है।"), "JA": ("選手は", "打つボールを。", "ボールを打つ。"),
  "KO": ("선수는", "친다 공을.", "공을 친다."), "TR": ("Oyuncu", "vuruyor topa.", "topa vuruyor.")},
 {"EN": ("The nurse", "brings the medicine.", "the medicine brings."), "FR": ("L'infirmière", "apporte le médicament.", "le médicament apporte."),
  "ES": ("La enfermera", "trae la medicina.", "la medicina trae."), "DE": ("Die Krankenschwester", "bringt das Medikament.", "das Medikament bringt."),
  "HI": ("नर्स", "लाती है दवा।", "दवा लाती है।"), "JA": ("看護師は", "持ってくる薬を。", "薬を持ってくる。"),
  "KO": ("간호사는", "가져온다 약을.", "약을 가져온다."), "TR": ("Hemşire", "getiriyor ilacı.", "ilacı getiriyor.")},
 {"EN": ("The artist", "paints the picture.", "the picture paints."), "FR": ("L'artiste", "peint le tableau.", "le tableau peint."),
  "ES": ("El artista", "pinta el cuadro.", "el cuadro pinta."), "DE": ("Der Künstler", "malt das Bild.", "das Bild malt."),
  "HI": ("कलाकार", "बनाता है चित्र।", "चित्र बनाता है।"), "JA": ("画家は", "描く絵を。", "絵を描く。"),
  "KO": ("화가는", "그린다 그림을.", "그림을 그린다."), "TR": ("Ressam", "çiziyor resmi.", "resmi çiziyor.")},
 {"EN": ("The worker", "builds the wall.", "the wall builds."), "FR": ("L'ouvrier", "construit le mur.", "le mur construit."),
  "ES": ("El obrero", "construye el muro.", "el muro construye."), "DE": ("Der Arbeiter", "baut die Mauer.", "die Mauer baut."),
  "HI": ("मज़दूर", "बनाता है दीवार।", "दीवार बनाता है।"), "JA": ("労働者は", "建てる壁を。", "壁を建てる。"),
  "KO": ("노동자는", "쌓는다 벽을.", "벽을 쌓는다."), "TR": ("İşçi", "yapıyor duvarı.", "duvarı yapıyor.")},
 {"EN": ("The grandmother", "knits the sweater.", "the sweater knits."), "FR": ("La grand-mère", "tricote le pull.", "le pull tricote."),
  "ES": ("La abuela", "teje el suéter.", "el suéter teje."), "DE": ("Die Oma", "strickt den Pullover.", "den Pullover strickt."),
  "HI": ("दादी", "बुनती है स्वेटर।", "स्वेटर बुनती है।"), "JA": ("おばあさんは", "編むセーターを。", "セーターを編む。"),
  "KO": ("할머니는", "짠다 스웨터를.", "스웨터를 짠다."), "TR": ("Nine", "örüyor kazağı.", "kazağı örüyor.")},
 {"EN": ("The pilot", "checks the map.", "the map checks."), "FR": ("Le pilote", "vérifie la carte.", "la carte vérifie."),
  "ES": ("El piloto", "revisa el mapa.", "el mapa revisa."), "DE": ("Der Pilot", "prüft die Karte.", "die Karte prüft."),
  "HI": ("पायलट", "देखता है नक्शा।", "नक्शा देखता है।"), "JA": ("パイロットは", "確認する地図を。", "地図を確認する。"),
  "KO": ("조종사는", "확인한다 지도를.", "지도를 확인한다."), "TR": ("Pilot", "kontrol ediyor haritayı.", "haritayı kontrol ediyor.")},
 {"EN": ("The guard", "watches the gate.", "the gate watches."), "FR": ("Le gardien", "surveille le portail.", "le portail surveille."),
  "ES": ("El guardia", "vigila la puerta.", "la puerta vigila."), "DE": ("Der Wächter", "bewacht das Tor.", "das Tor bewacht."),
  "HI": ("चौकीदार", "देखता है फाटक।", "फाटक देखता है।"), "JA": ("警備員は", "見張る門を。", "門を見張る。"),
  "KO": ("경비원은", "지킨다 문을.", "문을 지킨다."), "TR": ("Bekçi", "bekliyor kapıyı.", "kapıyı bekliyor.")},
 {"EN": ("The neighbor", "fixes the fence.", "the fence fixes."), "FR": ("Le voisin", "répare la clôture.", "la clôture répare."),
  "ES": ("El vecino", "arregla la cerca.", "la cerca arregla."), "DE": ("Der Nachbar", "repariert den Zaun.", "den Zaun repariert."),
  "HI": ("पड़ोसी", "ठीक करता है बाड़।", "बाड़ ठीक करता है।"), "JA": ("隣人は", "直す柵を。", "柵を直す。"),
  "KO": ("이웃은", "고친다 울타리를.", "울타리를 고친다."), "TR": ("Komşu", "tamir ediyor çiti.", "çiti tamir ediyor.")},
 {"EN": ("The sister", "sends the message.", "the message sends."), "FR": ("La sœur", "envoie le message.", "le message envoie."),
  "ES": ("La hermana", "envía el mensaje.", "el mensaje envía."), "DE": ("Die Schwester", "schickt die Nachricht.", "die Nachricht schickt."),
  "HI": ("बहन", "भेजती है संदेश।", "संदेश भेजती है।"), "JA": ("姉は", "送るメッセージを。", "メッセージを送る。"),
  "KO": ("여동생은", "보낸다 메시지를.", "메시지를 보낸다."), "TR": ("Abla", "gönderiyor mesajı.", "mesajı gönderiyor.")},
]

ORDER_LANGS = ["EN", "FR", "ES", "DE", "HI", "JA", "KO", "TR"]
ORDER_NATIVE = {"EN": "head_first", "FR": "head_first", "ES": "head_first", "DE": "head_first",
                "HI": "head_last", "JA": "head_last", "KO": "head_last", "TR": "head_last"}
N_SCENES_HALF = len(ORDER_SCENES) // 2   

def _split_of(scene_idx):
    return "derivation" if scene_idx < N_SCENES_HALF else "test"

# Cross-language clause bank (recipients; DE also appears here as the V2 main clause)
CLAUSE_BANK = []
for si, sc in enumerate(ORDER_SCENES):
    for lang in ORDER_LANGS:
        pre, vo, ov = sc[lang]
        CLAUSE_BANK.append(dict(lang=lang, scene=si, prefix=pre, cand_vo=vo, cand_ov=ov,
                                native=ORDER_NATIVE[lang], split=_split_of(si)))

# German in-language minimal pairs: SAME scenes, three frames.
#   V2  : main clause, verb second            -> native head-first   ("Der Mann | liest das Buch.")
#   SUB : 'dass'-clause, verb final           -> native head-last    ("Ich glaube, dass der Mann | das Buch liest.")
#   OB  : 'ob'-clause, verb final (NUISANCE)  -> native head-last; same order as SUB, different complementizer.
def _lc(s):
    return s[0].lower() + s[1:]

DE_FRAMES = []
for si, sc in enumerate(ORDER_SCENES):
    pre, vo, ov = sc["DE"]
    DE_FRAMES.append(dict(lang="DE", scene=si, kind="V2",  prefix=pre, cand_vo=vo, cand_ov=ov,
                          native="head_first", split=_split_of(si)))
    DE_FRAMES.append(dict(lang="DE", scene=si, kind="SUB", prefix="Ich glaube, dass " + _lc(pre), cand_vo=vo, cand_ov=ov,
                          native="head_last", split=_split_of(si)))
    DE_FRAMES.append(dict(lang="DE", scene=si, kind="OB",  prefix="Ich weiß nicht, ob " + _lc(pre), cand_vo=vo, cand_ov=ov,
                          native="head_last", split=_split_of(si)))

REQUIRED_SCENES = 8

def validate_order_bank(de_frames, clause_bank, langs_expected=tuple(ORDER_LANGS)):
    missing = []
    for split in ("derivation", "test"):
        n = len({f["scene"] for f in de_frames if f["split"] == split and f["kind"] == "V2"})
        if n < REQUIRED_SCENES:
            missing.append(f"DE_FRAMES/{split}: {n} scenes, need >= {REQUIRED_SCENES}")
        for lang in langs_expected:
            n = len([u for u in clause_bank if u["lang"] == lang and u["split"] == split])
            if n < REQUIRED_SCENES:
                missing.append(f"CLAUSE_BANK/{lang}/{split}: {n} scenes, need >= {REQUIRED_SCENES}")
    return missing

_missing_order = validate_order_bank(DE_FRAMES, CLAUSE_BANK)
ORDER_BANK_READY = (len(_missing_order) == 0)
if not ORDER_BANK_READY:
    print("ORDER item bank FAILS the count check:")
    for m in _missing_order:
        print("  -", m)
else:
    print(f"ORDER item bank passes the count check: {len(CLAUSE_BANK)} clause rows, {len(DE_FRAMES)} German frame rows.")
    print("Authored by me, not natively reviewed (HI/JA/KO/TR in particular) -- the clean-accuracy gate is the empirical check.")


ORDER item bank passes the count check: 192 clause rows, 72 German frame rows.
Authored by me, not natively reviewed (HI/JA/KO/TR in particular) -- the clean-accuracy gate is the empirical check.


In [6]:
import csv, time, math, os, json

# ---------------------------------------------------------------------------
# Batch machinery shared by every experiment below.
#   A "prep" holds, for one language/task/half, 2N rows: N rows with the class-A continuation
#   (gender: feminine; order: verb-object / head-first) and N rows with the class-B continuation.
#   S = logP(A) - logP(B) at the decision token. native_sign = +1 if the unit's native class is A else -1,
#   so the NATIVE margin is native_sign * S. Interventions push AWAY from the native class.
# ---------------------------------------------------------------------------
JOIN = {"JA": ""}   # every other language joins prefix and continuation with a single space

def unit_vec(v):
    return v / (np.linalg.norm(v) + 1e-12)

def build_prep(units, kind):
    kept, dropped = [], 0
    rowsA, rowsB, cA, cB, pos = [], [], [], [], []
    for u in units:
        j = JOIN.get(u["lang"], " ")
        if kind == "gender":
            tA, tB = u["prefix"] + j + u["fem"], u["prefix"] + j + u["masc"]
            native_is_A = (u["native"] == "fem")
            scene = u["concept"]
        else:
            tA, tB = u["prefix"] + j + u["cand_vo"], u["prefix"] + j + u["cand_ov"]
            native_is_A = (u["native"] == "head_first")
            scene = u["scene"]
        eA = encode_unit(u["prefix"], tA, tok)
        eB = encode_unit(u["prefix"], tB, tok)
        if (eA is None or eB is None or eA[1] != eB[1]
                or len(eA[0]) <= eA[1] + 1 or len(eB[0]) <= eB[1] + 1):
            dropped += 1          # counted, never silently excluded
            continue
        (idsA, p), (idsB, _) = eA, eB
        kept.append(dict(u, scene_id=scene, native_sign=(1.0 if native_is_A else -1.0)))
        rowsA.append(idsA); rowsB.append(idsB)
        cA.append(list(idsA[p + 1:])); cB.append(list(idsB[p + 1:])); pos.append(p)
    if not kept:
        return dict(N=0, units=[], n_dropped=dropped, kind=kind)
    pad = getattr(tok, "pad_token_id", 0) or 0
    ids, mask = collate(rowsA + rowsB, pad)
    return dict(N=len(kept), units=kept, n_dropped=dropped, ids=ids, mask=mask, pos=pos, pos2=pos + pos,
                candA=cA, candB=cB, native_sign=np.array([u["native_sign"] for u in kept]), kind=kind)

def subset_prep(prep, idx):
    idx = [int(i) for i in idx]; N = prep["N"]
    rows = idx + [N + i for i in idx]
    return dict(N=len(idx), units=[prep["units"][i] for i in idx], ids=prep["ids"][rows], mask=prep["mask"][rows],
                pos=[prep["pos"][i] for i in idx], pos2=[prep["pos"][i] for i in idx] * 2,
                candA=[prep["candA"][i] for i in idx], candB=[prep["candB"][i] for i in idx],
                native_sign=prep["native_sign"][idx], kind=prep["kind"], n_dropped=0)

def make_delta_hook(D, pos_list):
    """Adds the per-row vector D[i] at position pos_list[i] (D: [rows, d] float array)."""
    def fn(module, inp, out):
        is_tuple = isinstance(out, tuple)
        h = out[0] if is_tuple else out
        hc = h.clone()
        Dt = torch.tensor(D, dtype=torch.float32, device=hc.device)
        for i, p in enumerate(pos_list):
            hc[i, p, :] = (hc[i, p, :].float() + Dt[i]).to(hc.dtype)
        return (hc,) + out[1:] if is_tuple else hc
    return fn

def _score_rows(logits, cands, pos, off):
    out = np.zeros(len(cands))
    for i, cand in enumerate(cands):
        p, L = pos[i], len(cand)
        seg = logits[off + i, p:p + L, :]
        lp = torch.log_softmax(seg.float(), dim=-1)
        ar = torch.arange(L, device=lp.device)
        ct = torch.tensor(cand, device=lp.device)
        out[i] = float(lp[ar, ct].sum())
    return out

def run_prep(prep, steer=None, knock=None):
    """Margins S (A minus B) for every unit in the prep.
       steer: {layer: delta[N, d]}  -> adds delta[i] at unit i's decision token (both A and B rows)
       knock: {layer: unit_vector}  -> projects that direction out at every row's decision token"""
    dev = next(model.parameters()).device
    ids, mask = prep["ids"].to(dev), prep["mask"].to(dev)
    _clear_hooks()
    for L, delta in (steer or {}).items():
        D = np.concatenate([delta, delta], 0)
        HOOK_REGISTRY.append(LAYERS[L].register_forward_hook(make_delta_hook(D, prep["pos2"])))
    for L, vec in (knock or {}).items():
        HOOK_REGISTRY.append(LAYERS[L].register_forward_hook(make_intervention_hook("knockout", prep["pos2"], vec=vec)))
    try:
        with torch.no_grad():
            logits = model(input_ids=ids, attention_mask=mask, use_cache=False).logits
    finally:
        _clear_hooks()
    N = prep["N"]
    return _score_rows(logits, prep["candA"], prep["pos"], 0) - _score_rows(logits, prep["candB"], prep["pos"], N)

def ensure_clean(prep):
    if "S0" not in prep:
        prep["S0"] = run_prep(prep)
    return prep["S0"]

def get_states(prep):
    """[N, n_layers, d] decision-token residual states (taken from the A rows; prefix is shared with B rows)."""
    if "states" not in prep:
        N = prep["N"]
        prep["states"] = capture_states(prep["ids"][:N], prep["mask"][:N], prep["pos"])
    return prep["states"]

def clean_gate(prepD, prepT, cfg):
    ok_items = tot = 0
    for p in (prepD, prepT):
        if p["N"] == 0:
            continue
        S0 = ensure_clean(p)
        tot += p["N"]
        ok_items += int(((p["native_sign"] * S0) > 0).sum())
    acc = ok_items / tot if tot else 0.0
    ok = (acc >= cfg["gate_acc"]) and prepD["N"] >= cfg["gate_min_items"] and prepT["N"] >= cfg["gate_min_items"]
    return dict(acc=float(acc), n_deriv=prepD["N"], n_test=prepT["N"], ok=bool(ok),
                dropped=prepD.get("n_dropped", 0) + prepT.get("n_dropped", 0))

print("prep / run / gate machinery ready")


prep / run / gate machinery ready


In [7]:
def diff_of_means_direction(states_class_a, states_class_b):
    """states_class_a/b: [N, d] at one layer. Returns unnormalized mean-difference vector."""
    return states_class_a.mean(0) - states_class_b.mean(0)

def in_language_curve(states_all_layers_a, states_all_layers_b, margin_fn):
    """states_*: [N, n_layers, d]. Returns per-layer in-language effect using margin_fn,
    which scores how well each layer's diff-of-means direction separates the two classes
    on a held-out slice (caller passes derivation-internal held-out states)."""
    n_layers = states_all_layers_a.shape[1]
    effect = np.zeros(n_layers)
    for li in range(n_layers):
        d = diff_of_means_direction(states_all_layers_a[:, li, :], states_all_layers_b[:, li, :])
        effect[li] = margin_fn(d, li)
    return effect

def select_window(effect_curve, cfg):
    """Layers where effect >= window_frac_of_max * max(effect). Returns sorted layer indices.
    This selection happens on the DERIVATION half only -- never on cross-language results --
    per the frozen rule in the handoff (Section 5.1)."""
    if effect_curve.max() <= 0:
        return []
    thresh = cfg["window_frac_of_max"] * effect_curve.max()
    return sorted(int(i) for i in np.where(effect_curve >= thresh)[0])

print("diff-of-means window-selection ready")


diff-of-means window-selection ready


In [8]:
def spsa_fit_direction(init_vec, loss_fn, cfg, holdout_loss_fn=None, rng=None):
    """Zeroth-order (SPSA) fit of a unit vector minimising loss_fn(vec). Forward passes only.

    FIX vs the first draft: the probe is delta/sqrt(d), i.e. a UNIT-norm random sign vector scaled by c,
    so a probe of radius c is the same size in 8 dims or 3584 dims. (The first draft probed with c*delta, whose
    norm is c*sqrt(d) -- ~3 in a 3584-dim residual -- which would have swamped the direction. It was never
    caught because the self-test uses d=8.) The update is a normalised sign step with 1/sqrt(t) decay.
    In the pipeline this runs in a SMALL subspace (<= ~9 dims), where SPSA is well-conditioned.

    holdout_loss_fn is used for early stopping / best-iterate tracking only. Returns (unit vector, loss history).
    """
    rng = rng or np.random.default_rng(0)
    nrm = lambda v: v / (np.linalg.norm(v) + 1e-12)
    d = init_vec.shape[0]
    theta = nrm(init_vec)
    c = cfg["das_spsa_c"]
    eta0 = cfg["das_spsa_lr"] * c
    best_h = holdout_loss_fn(theta) if holdout_loss_fn is not None else None
    best_theta, patience, history = theta.copy(), cfg["das_patience"], []
    for t in range(cfg["das_spsa_steps"]):
        delta = rng.choice([-1.0, 1.0], size=d) / np.sqrt(d)
        lp = loss_fn(nrm(theta + c * delta))
        lm = loss_fn(nrm(theta - c * delta))
        eta = eta0 / np.sqrt(1.0 + t / 25.0)
        theta = nrm(theta - eta * np.sign(lp - lm) * delta)
        history.append(float(0.5 * (lp + lm)))
        if holdout_loss_fn is not None:
            h = holdout_loss_fn(theta)
            if h < best_h - 1e-6:
                best_h, best_theta, patience = h, theta.copy(), cfg["das_patience"]
            else:
                patience -= 1
                if patience <= 0:
                    break
    final = best_theta if holdout_loss_fn is not None else theta
    return nrm(final), history

print("SPSA direction-fitting ready (probe radius is dimension-independent)")


SPSA direction-fitting ready (probe radius is dimension-independent)


In [9]:
def calibrate_alpha_star(direction, layer_idx, deriv_units, sigma, cfg, flip_eval_fn):
    """Smallest alpha in cfg['alpha_grid'] at which the SOURCE language's own derivation-half flip rate reaches
    cfg['alpha_target_flip'] (falls back to suff_min_flip).

    Why a separate target (rule-v2): if alpha* is the smallest alpha that just reaches 0.5 on the derivation half and the
    test-half sufficiency gate is ALSO 0.5, a direction with a true flip rate near 0.5 fails the gate about half the time
    purely by regression to the mean. Calibrating to a higher derivation target (0.65) gives the test half headroom.
    It applies identically to gender and order. The gate itself (suff_min_flip, test half) is unchanged."""
    target = cfg.get("alpha_target_flip", cfg["suff_min_flip"])
    for alpha in cfg["alpha_grid"]:
        fr = flip_eval_fn(direction, layer_idx, alpha, sigma)
        if fr >= target:
            return alpha, fr
    return cfg["alpha_grid"][-1], flip_eval_fn(direction, layer_idx, cfg["alpha_grid"][-1], sigma)

print("alpha* calibration ready")


alpha* calibration ready


In [10]:
# ---------------------------------------------------------------------------
# Effects, directions, window selection, DAS-lite fit, necessity, per-source processing.
# ---------------------------------------------------------------------------
def class_direction(X, sign):
    """Unit direction pointing TOWARD class A (gender: fem minus masc; order: head-first minus head-last)."""
    return unit_vec(X[sign > 0].mean(0) - X[sign < 0].mean(0))

def orth_basis(u0, X_train, k):
    Xc = X_train - X_train.mean(0, keepdims=True)
    V = np.zeros((X_train.shape[1], 0))
    if Xc.shape[0] > 1 and k > 0:
        _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
        V = Vt[:k].T
    Q, _ = np.linalg.qr(np.concatenate([u0[:, None], V], axis=1))
    if float(Q[:, 0] @ u0) < 0:
        Q[:, 0] *= -1
    return Q

def steer_deltas(sign, alpha, sigma, u):
    """Push AWAY from each unit's native class: delta_i = -native_sign_i * alpha * sigma * u_A."""
    return -(sign[:, None]) * float(alpha) * float(sigma) * np.atleast_2d(u)

def effect_from_margins(S0, S1, sign):
    m0, m1 = sign * S0, sign * S1
    ok = m0 > 0                                   # only clean-correct units can "lose" their native preference
    return dict(idx=np.where(ok)[0], flip=(m1[ok] < 0).astype(float), shift=(m0 - m1)[ok], m0=m0[ok])

def frac_removed(e):
    """Fraction of the native margin removed, as a RATIO OF MEANS (a mean of per-item ratios explodes on items whose
    clean margin is ~0). 1.0 = margin fully removed; negative = intervention increased it."""
    return float(e["shift"].sum() / max(float(e["m0"].sum()), 1e-6)) if len(e["shift"]) else 0.0

def mean_or_zero(x):
    return float(np.mean(x)) if len(x) else 0.0

def eff_steer(prep, L, u, alpha, sigma):
    S0 = ensure_clean(prep)
    S1 = run_prep(prep, steer={L: steer_deltas(prep["native_sign"], alpha, sigma, u)})
    return effect_from_margins(S0, S1, prep["native_sign"])

def flip_rate(prep, L, u, alpha, sigma):
    return mean_or_zero(eff_steer(prep, L, u, alpha, sigma)["flip"])

def flip_rate_both_signs(prep, L, u, alpha, sigma):
    return max(flip_rate(prep, L, u, alpha, sigma), flip_rate(prep, L, -u, alpha, sigma))

def control_flips(prep, L, alpha, sigma, dirs):
    return np.array([flip_rate(prep, L, d, alpha, sigma) for d in dirs])

def group_by_scene(prep, e, key):
    groups = {}
    for i, v in zip(e["idx"], e[key]):
        groups.setdefault(prep["units"][int(i)]["scene_id"], []).append(v)
    return [np.array(v) for v in groups.values()]

def loo_curve(prepD, statesD, layers, alpha):
    """Leave-one-out diff-of-means steering: item i is pushed with the direction computed WITHOUT item i."""
    sign, N = prepD["native_sign"], prepD["N"]
    S0 = ensure_clean(prepD)
    out = []
    for L in layers:
        X = statesD[:, L, :]
        sigma = float(np.linalg.norm(X, axis=-1).mean())
        U = np.zeros_like(X)
        for i in range(N):
            m = np.arange(N) != i
            U[i] = class_direction(X[m], sign[m]) if ((sign[m] > 0).any() and (sign[m] < 0).any()) else class_direction(X, sign)
        e = effect_from_margins(S0, run_prep(prepD, steer={L: steer_deltas(sign, alpha, sigma, U)}), sign)
        out.append(dict(layer=int(L), shift=mean_or_zero(e["shift"]), flip=mean_or_zero(e["flip"]), sigma=sigma))
    return out

def fit_layer_direction(prepD, statesD, L, cfg, seed):
    """DAS-lite at one layer: SPSA inside span{diff-of-means axis, top-k derivation-state PCs}, loss = softplus of the
    patched NATIVE margin (small when the native preference is flipped). The fitted direction replaces the
    diff-of-means direction ONLY if it beats it on held-out derivation items by das_min_improvement nats."""
    rng = np.random.default_rng(seed * 1000 + L)
    sign, N = prepD["native_sign"], prepD["N"]
    X = statesD[:, L, :]
    sigma = float(np.linalg.norm(X, axis=-1).mean())
    u0_all = class_direction(X, sign)
    res = dict(u=u0_all, u0=u0_all, sigma=sigma, used_fit=False, hold_init=float("nan"), hold_fit=float("nan"), steps=0)
    posA, negB = np.where(sign > 0)[0], np.where(sign < 0)[0]
    if len(posA) < 2 or len(negB) < 2:
        return res
    nh = max(2, int(round(cfg["das_holdout_frac"] * N)))
    hold = [int(rng.choice(posA)), int(rng.choice(negB))]
    rest = [int(i) for i in rng.permutation(N) if int(i) not in hold]
    hold += rest[: max(0, nh - 2)]
    train = [i for i in range(N) if i not in hold]
    st, sh = sign[train], sign[hold]
    if not ((st > 0).any() and (st < 0).any()):
        return res
    Q = orth_basis(class_direction(X[train], st), X[train], cfg["das_subspace_k"])
    pt, ph = subset_prep(prepD, train), subset_prep(prepD, hold)
    def mk(p, s):
        def loss(w):
            u = Q @ (w / (np.linalg.norm(w) + 1e-12))
            S1 = run_prep(p, steer={L: steer_deltas(s, cfg["das_alpha_fit"], sigma, u)})
            return float(np.logaddexp(0.0, s * S1).mean())
        return loss
    ltr, lho = mk(pt, st), mk(ph, sh)
    w0 = np.zeros(Q.shape[1]); w0[0] = 1.0
    wfit, hist = spsa_fit_direction(w0, ltr, cfg, holdout_loss_fn=lho, rng=rng)
    h0, h1 = lho(w0), lho(wfit)
    used = bool(h1 < h0 - cfg["das_min_improvement"])
    res.update(used_fit=used, hold_init=float(h0), hold_fit=float(h1), steps=len(hist))
    if used:
        res["u"] = unit_vec(Q @ wfit)
    return res

def knockout_effect(prep, dirs_by_layer):
    return effect_from_margins(ensure_clean(prep), run_prep(prep, knock=dirs_by_layer), prep["native_sign"])

def random_knockout_baseline(prep, layers, d, n, rng):
    fr = []
    for _ in range(n):
        e = knockout_effect(prep, {L: unit_vec(rng.normal(size=d)) for L in layers})
        fr.append(frac_removed(e))
    return float(np.mean(fr)) if fr else 0.0

def process_source(ctx, cfg, seed=0):
    """Everything that depends only on the SOURCE language/task, using the derivation half to choose anything
    (window, directions, layer, alpha*) and the test half only to evaluate."""
    pD, pT = ctx["D"], ctx["T"]
    statesD = get_states(pD)
    layers = list(range(0, N_LAYERS, cfg["LAYER_STRIDE"]))
    curve = loo_curve(pD, statesD, layers, cfg["alpha_probe"])
    win_idx = select_window(np.array([c["shift"] for c in curve]), cfg)
    window = [layers[i] for i in win_idx]
    out = dict(task=ctx["task"], lang=ctx["lang"], curve=curve, window=window, ok=False, reason="")
    if not window:
        out["reason"] = "no_window: leave-one-out diff-of-means never moved the readout"
        return out
    fits = {L: fit_layer_direction(pD, statesD, L, cfg, seed) for L in window}
    dshift = {}
    for L in window:
        dshift[L] = mean_or_zero(eff_steer(pD, L, fits[L]["u"], cfg["alpha_probe"], fits[L]["sigma"])["shift"])
    Lstar = max(window, key=lambda L: dshift[L])
    f = fits[Lstar]
    alpha_star, _ = calibrate_alpha_star(f["u"], Lstar, pD["units"], f["sigma"], cfg,
                                         lambda u, L, a, s: flip_rate(pD, L, u, a, s))
    eT = eff_steer(pT, Lstar, f["u"], alpha_star, f["sigma"])
    flip_src, shift_src = mean_or_zero(eT["flip"]), mean_or_zero(eT["shift"])
    flip_src_u0 = flip_rate(pT, Lstar, f["u0"], alpha_star, f["sigma"])   # plain diff-of-means direction, same layer/alpha
    dose_self = [dict(alpha=float(al), flip=flip_rate(pT, Lstar, f["u"], al, f["sigma"])) for al in cfg["alpha_grid"]]
    suff_ok = bool(len(eT["flip"]) >= cfg["min_correct_items"] and flip_src >= cfg["suff_min_flip"])
    # necessity: knock the direction out across the whole window (single-layer ablation is undone by later layers)
    ek = knockout_effect(pT, {L: fits[L]["u"] for L in window})
    nec_frac = frac_removed(ek)
    rand_frac = random_knockout_baseline(pT, window, hidden_dim(), min(20, cfg["n_random_dirs"]),
                                         np.random.default_rng(seed + 123))
    nec_ok = bool(len(ek["shift"]) >= cfg["min_correct_items"] and nec_frac >= cfg["necessity_min_drop"]
                  and nec_frac >= cfg["necessity_vs_random_ratio"] * max(rand_frac, 0.01))
    XA, XB = statesD[pD["native_sign"] > 0][:, Lstar, :], statesD[pD["native_sign"] < 0][:, Lstar, :]
    perm = [unit_vec(v) for v in permutation_directions(XA, XB, cfg["n_perm_dirs"], np.random.default_rng(seed + 7))]
    # v5: layers where the SOURCE direction reaches alpha_target_flip on its derivation half at alpha* (same headroom logic as rule-v2:
    # test-half gate 0.5 vs derivation target 0.65, otherwise regression to the mean fails the gate ~half the time) are eligible for recipient
    # injection; each gets an in-language TEST evaluation so R at layer L divides by the source's own flip at layer L.
    elig, layer_eval = [], {}
    for L in window:
        ok_L = True if L == Lstar else bool(flip_rate(pD, L, fits[L]["u"], alpha_star, fits[L]["sigma"]) >= cfg["alpha_target_flip"])
        if ok_L:
            elig.append(L)
            eL = eff_steer(pT, L, fits[L]["u"], alpha_star, fits[L]["sigma"])
            layer_eval[L] = dict(flip=mean_or_zero(eL["flip"]), shift=mean_or_zero(eL["shift"]), n=len(eL["flip"]),
                                 groups=group_by_scene(pT, eL, "flip"))
    out.update(elig_layers=elig, layer_eval=layer_eval, pD=pD, pT=pT)
    out.update(flip_src_u0=flip_src_u0, dose_self=dose_self, alpha_at_floor=bool(alpha_star == cfg['alpha_grid'][0]), ok=True, fits=fits, Lstar=int(Lstar), alpha=float(alpha_star), u_star=f["u"], u0=f["u0"], sigma=f["sigma"],
               flip_src=flip_src, shift_src=shift_src, n_src=len(eT["flip"]), src_groups=group_by_scene(pT, eT, "flip"),
               suff_ok=suff_ok, nec_ok=nec_ok, nec_frac=nec_frac, nec_flip=mean_or_zero(ek["flip"]), rand_knock_frac=rand_frac,
               perm_dirs=perm, deriv_shift=dshift, statesD_L=statesD[:, Lstar, :])
    return out

print("effects / fit / necessity machinery ready")


effects / fit / necessity machinery ready


In [11]:
def random_directions(d, n, rng=None):
    rng = rng or np.random.default_rng(1)
    V = rng.normal(size=(n, d))
    V /= np.linalg.norm(V, axis=1, keepdims=True) + 1e-8
    return V

def permutation_directions(states_a, states_b, n, rng=None):
    """Re-derive diff-of-means directions after shuffling the class labels across the
    pooled (a+b) items, n times -- null distribution for 'direction is just item/concept
    noise', per Section 5.3.2."""
    rng = rng or np.random.default_rng(2)
    pooled = np.concatenate([states_a, states_b], axis=0)
    n_a = states_a.shape[0]
    out = []
    for _ in range(n):
        idx = rng.permutation(pooled.shape[0])
        a_idx, b_idx = idx[:n_a], idx[n_a:]
        out.append(pooled[a_idx].mean(0) - pooled[b_idx].mean(0))
    return np.stack(out)

def cue_only_direction(states_lang_x_neutral, states_lang_y_neutral):
    """A REAL language-identity direction: mean state on a NEUTRAL task (unrelated to
    gender/order) in language X minus language Y. Used as a cue-only falsifier per the
    triangulation standard -- stronger than norm-matched random noise, because it
    specifically carries 'this came from a different language' rather than generic noise.
    If patching THIS moves the gender/order readout about as much as the real direction
    does, the apparent transfer may just be 'patching anything language-flavored nudges
    this readout', which random-direction controls would not catch."""
    return states_lang_x_neutral.mean(0) - states_lang_y_neutral.mean(0)

def specificity_2x2(effect_own_task, effect_cross_task, min_ratio=3.0):
    """A shared code should show own-task effect >= min_ratio * cross-task effect
    (Section 5.3.4): applying d_g to the ORDER readout and d_o to the GENDER readout."""
    if effect_cross_task <= 1e-8:
        return effect_own_task > 0, float('inf')
    ratio = effect_own_task / effect_cross_task
    return ratio >= min_ratio, ratio

print("control functions ready: random, permutation, nuisance-direction (reuse sufficiency_effect),"
      " 2x2 specificity, and a real cue-only language-ID falsifier")


control functions ready: random, permutation, nuisance-direction (reuse sufficiency_effect), 2x2 specificity, and a real cue-only language-ID falsifier


In [12]:
def bootstrap_ci(values_by_scene, stat_fn, n_boot, rng=None, alpha=0.05):
    """Resample SCENES jointly (not items), per Section 5.2's convention.
    values_by_scene: list of per-scene value arrays (already grouped by scene)."""
    rng = rng or np.random.default_rng(3)
    n = len(values_by_scene)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        sample = [values_by_scene[i] for i in idx]
        boots.append(stat_fn(sample))
    boots = np.sort(np.array(boots))
    lo = boots[int((alpha / 2) * n_boot)]
    hi = boots[int((1 - alpha / 2) * n_boot) - 1]
    return float(np.mean(boots)), float(lo), float(hi)

def transfer_ratio(effect_recipient, effect_source_inlang):
    if effect_source_inlang <= 1e-8:
        return None  # source gate should have already failed upstream of this
    return effect_recipient / effect_source_inlang

print("bootstrap + transfer-ratio functions ready (scene-level resampling)")

def bootstrap_ratio_ci(recip_groups, src_groups, n_boot, rng=None, alpha=0.05):
    """95% CI of R = mean(recipient flips) / mean(source in-language flips); scenes resampled independently
    within the recipient set and within the source set (they are different scenes/languages)."""
    rng = rng or np.random.default_rng(5)
    if not recip_groups or not src_groups:
        return (None, None, None)
    boots = []
    for _ in range(n_boot):
        r = np.concatenate([recip_groups[i] for i in rng.integers(0, len(recip_groups), len(recip_groups))])
        s = np.concatenate([src_groups[i] for i in rng.integers(0, len(src_groups), len(src_groups))])
        if s.mean() > 1e-8:
            boots.append(r.mean() / s.mean())
    if not boots:
        return (None, None, None)
    b = np.sort(np.array(boots))
    return float(b.mean()), float(b[int(alpha / 2 * len(b))]), float(b[min(len(b) - 1, int((1 - alpha / 2) * len(b)))])


def exact_binom_upper(k, n, alpha=0.05):
    """Clopper-Pearson two-sided (1-alpha) UPPER bound on a flip probability given k flips in n clean-correct items.
    0 of 12 -> 0.265: the cell cannot exclude a true flip rate of ~0.25. (Bootstrap CIs of an all-zero sample are [0,0] -- an artifact, not precision.)"""
    if n <= 0:
        return 1.0
    k = int(round(k))
    if k >= n:
        return 1.0
    lo, hi = 0.0, 1.0
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        cdf = sum(math.comb(n, j) * mid ** j * (1 - mid) ** (n - j) for j in range(k + 1))
        if cdf > alpha / 2:
            lo = mid
        else:
            hi = mid
    return float(hi)

def dose_ed50(alphas, flips, level=0.5):
    """Smallest alpha at which the flip rate first reaches `level`, log-linear interpolation against the previous grid
    point. None if never reached on the grid. Replaces 'flip at one alpha' as the sensitivity measure (flip saturates at 1.0)."""
    prev = None
    for a, f in sorted(zip(alphas, flips)):
        if f >= level:
            if prev is None:
                return float(a)
            a0, f0 = prev
            t = (level - f0) / max(f - f0, 1e-9)
            return float(math.exp(math.log(a0) + t * (math.log(a) - math.log(a0))))
        prev = (a, f)
    return None

def sensitivity_ratio(ed_src, ed_rec):
    """ED50(source in its own language) / ED50(recipient). 1 = same push needed; >1 recipient MORE sensitive;
    0.0 = recipient never reaches 0.5 anywhere on the grid (a wall up to the largest alpha); None = source never reached 0.5."""
    if ed_src is None:
        return None
    if ed_rec is None:
        return 0.0
    return float(ed_src / ed_rec)

print("exact binomial bound + ED50 / sensitivity ratio ready")


bootstrap + transfer-ratio functions ready (scene-level resampling)
exact binomial bound + ED50 / sensitivity ratio ready


In [13]:
def classify_cell(suff_ok, necessity_ok, controls_ok, R, R_ci, cfg, R_ub=None):
    """RULE-V3. Source sufficiency + necessity gate every cell. Controls gate ONLY a cell that claims transfer
    (R >= transfer_min): a control failure on a cell that claims nothing does not bear on the claim (run 3 lost
    DE->FR and ES->FR/HI->FR to the absolute 0.10 rule that way). A WALL additionally needs power: the exact 95%
    upper bound of R must be <= wall_ub_max, otherwise the cell is WALL_LOWPOWER (it cannot exclude transfer)."""
    if not (suff_ok and necessity_ok):
        return "UNUSABLE"
    if R is None:
        return "NO_CEILING"
    if R >= cfg["transfer_min"]:
        return "TRANSFER" if controls_ok else "UNUSABLE"
    if R <= cfg["wall_max"]:
        if R_ub is not None and R_ub > cfg["wall_ub_max"]:
            return "WALL_LOWPOWER"
        return "WALL"
    return "AMBIGUOUS"

def verdict(cell_report, cfg):
    """cell_report: list of dicts with at least {'task': 'gender'|'order', 'recipient':...,
    'class': one of TRANSFER/WALL/AMBIGUOUS/NO_CEILING/UNUSABLE, 'R':..., 'R_ci': (lo,hi)}.
    Mirrors the Section 5.4 decision tree, now requiring the necessity+sufficiency gates
    baked into classify_cell (above) to have already passed for every cell counted here."""
    gender = [c for c in cell_report if c["task"] == "gender" and c["recipient"] in ("FR", "ES")]
    order  = [c for c in cell_report if c["task"] == "order"  and c["recipient"] in ("FR", "ES")]

    if any(c["class"] == "UNUSABLE" for c in gender + order):
        return "INSTRUMENT_INADEQUATE: at least one required FR/ES cell is UNUSABLE (gate, sufficiency, necessity, or control failure)."
    if len(gender) < 2 or len(order) < 2:
        return "INCONCLUSIVE: fewer than 2 usable FR/ES cells for gender and/or order."

    def ok_transfer(cells):
        return all(c["class"] == "TRANSFER" for c in cells)
    def ok_wall(cells):
        return all(c["class"] == "WALL" for c in cells)
    def cis_separate(cells_a, cells_b):
        a_lo = min(c["R_ci"][0] for c in cells_a); a_hi = max(c["R_ci"][1] for c in cells_a)
        b_lo = min(c["R_ci"][0] for c in cells_b); b_hi = max(c["R_ci"][1] for c in cells_b)
        return (a_lo > b_hi) or (b_lo > a_hi)

    order_ge_gender = all(
        o["R"] is not None and g["R"] is not None and o["R"] >= g["R"]
        for o, g in zip(sorted(order, key=lambda c: c["recipient"]), sorted(gender, key=lambda c: c["recipient"]))
    )

    # Priority matters here and is itself part of the frozen rule (not an incidental
    # implementation detail): INVERSION_SUPPORTED and BOTH_SHARED are checked first because
    # they're the most specific/determinate findings. REVERSED is checked before BOTH_LOW
    # because BOTH_LOW only inspects gender's own R -- without this ordering, a clean
    # order-beats-gender result (gender WALL, order TRANSFER, CIs separate) would be caught
    # by BOTH_LOW's "gender R <= wall_max" clause before REVERSED ever got evaluated, which
    # would silently discard the more informative REVERSED finding. Caught by the toy-harness
    # branch-forcing self-test below -- do not reorder these without re-running that test.
    if ok_transfer(gender) and ok_wall(order) and (not cfg["ci_separation_required"] or cis_separate(gender, order)):
        return "INVERSION_SUPPORTED: gender transfers, order does not, CIs separate, controls and 2x2 specificity passed."
    if ok_transfer(gender) and ok_transfer(order):
        return "BOTH_SHARED: both gender and order transfer -- not the inversion."
    if order_ge_gender and cis_separate(order, gender):
        return "REVERSED: order transfers as much as or more than gender, CIs separate."
    if (not ok_transfer(gender)) and all(c["R"] is not None and c["R"] <= cfg["wall_max"] for c in gender):
        return "BOTH_LOW: gender does not transfer either -- the contrast fails."
    if any(c["class"] == "WALL_LOWPOWER" for c in gender + order):
        return "INCONCLUSIVE_LOWPOWER: at least one primary cell is a WALL whose exact upper bound cannot exclude transfer -- more items needed, not a wall."
    return "INCONCLUSIVE: no branch condition met -- report cell-level results, no post-hoc rule changes."

print("verdict logic ready")


verdict logic ready


In [14]:
# (1) additive no-op at alpha=0
# (2) knockout-of-zero-direction no-op
# (3) hooks are removed after every call (registry empty post-call)
# (4) SPSA loss should (on average) decrease on a toy convex-ish target
# These run against a TINY synthetic batch built straight from the tokenizer -- independent
# of the gender/order item banks, so they validate the intervention mechanics on their own,
# before any real GENDER_UNITS/DE_FRAMES data is filled in.

def _self_test_batch(n=3):
    texts = ["hello world test", "a second example", "and one more"][:n]
    ids_list = [tok(t)["input_ids"] for t in texts]
    pad_id = getattr(tok, "pad_token_id", 0) or 0
    ids, mask = collate(ids_list, pad_id)
    pos_list = [len(x) - 1 for x in ids_list]
    return ids, mask, pos_list

def self_test_hooks_cleared():
    assert len(HOOK_REGISTRY) == 0, "hook registry not empty after a call -- hooks are leaking"

def self_test_add_noop():
    ids, mask, pos_list = _self_test_batch()
    d = hidden_dim()
    base = clean_forward(ids, mask)
    patched = forward_with_intervention(ids, mask, 0, pos_list, "add", vec=np.ones(d), alpha=0.0, sigma=1.0)
    bn = base.detach().float().cpu().numpy() if hasattr(base, "numpy") else base
    pn = patched.detach().float().cpu().numpy() if hasattr(patched, "numpy") else patched
    delta = float(np.abs(bn - pn).max())
    assert delta < 1e-4, f"alpha=0 add was not a no-op, max|delta|={delta}"
    self_test_hooks_cleared()
    return delta

def self_test_knockout_zero_noop():
    ids, mask, pos_list = _self_test_batch()
    d = hidden_dim()
    base = clean_forward(ids, mask)
    patched = forward_with_intervention(ids, mask, 0, pos_list, "knockout", vec=np.zeros(d))
    bn = base.detach().float().cpu().numpy() if hasattr(base, "numpy") else base
    pn = patched.detach().float().cpu().numpy() if hasattr(patched, "numpy") else patched
    delta = float(np.abs(bn - pn).max())
    assert delta < 1e-4, f"knockout of a zero direction was not a no-op, max|delta|={delta}"
    self_test_hooks_cleared()
    return delta

def self_test_knockout_orthogonalizes():
    """After knockout along u, the component of the patched state along u should be ~0."""
    ids, mask, pos_list = _self_test_batch()
    d = hidden_dim()
    rng = np.random.default_rng(7)
    u = rng.normal(size=d); u /= np.linalg.norm(u)
    states_before = capture_states(ids, mask, pos_list)[:, 0, :]  # layer 0, [N, d]
    forward_with_intervention(ids, mask, 0, pos_list, "knockout", vec=u)
    # re-capture AFTER the (already-cleared) hook ran once more, this time keeping it live:
    _clear_hooks()
    hook = LAYERS[0].register_forward_hook(make_intervention_hook("knockout", pos_list, vec=u))
    HOOK_REGISTRY.append(hook)
    try:
        captured = {}
        def grab(mod, inp, out):
            captured["h"] = (out[0] if isinstance(out, tuple) else out).detach().float().cpu()
        h2 = LAYERS[0].register_forward_hook(grab)
        HOOK_REGISTRY.append(h2)
        with torch.no_grad():
            model(input_ids=ids, attention_mask=mask, use_cache=False)
    finally:
        _clear_hooks()
    after = captured["h"].numpy() if hasattr(captured["h"], "numpy") else captured["h"]
    resid = max(abs(float(after[i, pos_list[i], :] @ u)) for i in range(len(pos_list)))
    assert resid < 0.05, f"knockout left a residual component along u of {resid}"
    return resid

def self_test_spsa_decreases(d=8, steps=60, seed=0):
    rng = np.random.default_rng(seed)
    target = rng.normal(size=d); target /= np.linalg.norm(target)
    def loss_fn(v):
        v = v / (np.linalg.norm(v) + 1e-8)
        return float(1.0 - (v @ target))  # minimized when v aligns with target
    init = rng.normal(size=d)
    cfg_small = dict(das_spsa_steps=steps, das_spsa_c=0.1, das_spsa_lr=0.5,
                      das_patience=steps, das_early_stop_frac=0.0)
    direction, history = spsa_fit_direction(init, loss_fn, cfg_small, rng=rng)
    assert history[-1] < history[0], f"SPSA loss did not decrease: start={history[0]}, end={history[-1]}"
    return history[0], history[-1]

_d1 = self_test_add_noop()
_d2 = self_test_knockout_zero_noop()
_d3 = self_test_knockout_orthogonalizes()
_h0, _h1 = self_test_spsa_decreases()
print(f"self-test add no-op:            max|delta|   = {_d1:.2e}")
print(f"self-test knockout-zero no-op:  max|delta|   = {_d2:.2e}")
print(f"self-test knockout orthogonal:  residual     = {_d3:.2e}")
print(f"self-test SPSA converges:       loss {_h0:.4f} -> {_h1:.4f}")
print("SELF-TESTS PASSED")


self-test add no-op:            max|delta|   = 0.00e+00
self-test knockout-zero no-op:  max|delta|   = 0.00e+00
self-test knockout orthogonal:  residual     = 3.45e-02
self-test SPSA converges:       loss 1.2458 -> 0.6359
SELF-TESTS PASSED


In [15]:
# ---------------------------------------------------------------------------
# FULL RUN. Order of operations (everything that CHOOSES something uses the derivation half only):
#   1. build + gate every language/task (clean accuracy >= gate_acc, >= gate_min_items usable per half)
#   2. per source (gender: FR/ES/HI/AR that pass the gate; order: DE): LOO diff-of-means curve -> window ->
#      DAS-lite fit per window layer -> L* -> alpha* -> in-language SUFFICIENCY (test) and NECESSITY (test, band knockout)
#   3. per recipient (all gated languages except the source, test half only): cross-language flip, R, bootstrap CI,
#      controls (random, label-permutation, real language-ID cue, order-nuisance), 2x2 specificity, recipient-gender split
#   4. classify cells, assemble verdict from the four PRIMARY cells, write CSV/JSON/PNG
# ---------------------------------------------------------------------------
GENDER_LANGS = ["FR", "ES", "HI", "AR"]
ROMANCE = {"FR", "ES"}
BOTH_TASK_LANGS = {"FR", "ES", "HI"}       # languages that have BOTH a gender and an order readout (2x2 specificity)
PRIMARY = {"gender": [("ES", "FR"), ("FR", "ES")], "order": [("DE", "FR"), ("DE", "ES")]}   # (source, recipient)

NEUTRAL = {   # neutral sentence openers: no gender / word-order content; used only for the language-ID cue falsifier
    "EN": ["Today the weather is", "My name is", "The meeting starts at", "I live in", "Please close"],
    "FR": ["Aujourd'hui, le temps est", "Mon nom est", "La réunion commence à", "J'habite à", "Veuillez fermer"],
    "ES": ["Hoy el tiempo es", "Mi nombre es", "La reunión empieza a", "Vivo en", "Por favor cierra"],
    "DE": ["Heute ist das Wetter", "Mein Name ist", "Die Besprechung beginnt um", "Ich wohne in", "Bitte schließe"],
    "HI": ["आज मौसम", "मेरा नाम", "बैठक शुरू होती है", "मैं यहाँ", "कृपया बंद करें"],
    "JA": ["今日の天気は", "私の名前は", "会議は", "私は東京に", "どうぞ"],
    "KO": ["오늘 날씨는", "제 이름은", "회의는", "저는 서울에", "문을"],
    "TR": ["Bugün hava", "Benim adım", "Toplantı", "Ben İstanbul'da", "Lütfen"],
    "AR": ["الطقس اليوم", "اسمي", "الاجتماع يبدأ", "أسكن في", "من فضلك أغلق"],
}

def neutral_states(lang):
    ids_list = [tok(t)["input_ids"] for t in NEUTRAL[lang]]
    ids, mask = collate(ids_list, getattr(tok, "pad_token_id", 0) or 0)
    return capture_states(ids, mask, [len(x) - 1 for x in ids_list])

def prepare_language(task, lang):
    if task == "gender":
        units = [u for u in GENDER_UNITS if u["lang"] == lang]
    elif lang == "DE":
        units = [f for f in DE_FRAMES if f["kind"] in ("V2", "SUB")]
    else:
        units = [u for u in CLAUSE_BANK if u["lang"] == lang]
    pD = build_prep([u for u in units if u["split"] == "derivation"], task)
    pT = build_prep([u for u in units if u["split"] == "test"], task)
    ctx = dict(task=task, lang=lang, D=pD, T=pT)
    if pD["N"] and pT["N"]:
        ctx["gate"] = clean_gate(pD, pT, CONFIG)
    else:
        ctx["gate"] = dict(acc=0.0, n_deriv=pD["N"], n_test=pT["N"], ok=False, dropped=pD.get("n_dropped", 0) + pT.get("n_dropped", 0))
    if task == "order" and lang == "DE":
        ob = [f for f in DE_FRAMES if f["kind"] == "OB" and f["split"] == "derivation"]
        ctx["OB_D"] = build_prep(ob, "order")
    return ctx

def pipeline_self_tests(ctx):
    p = ctx["T"]; N = p["N"]
    S0 = ensure_clean(p)
    S_noop = run_prep(p, steer={0: np.zeros((N, hidden_dim()))})
    S_kz = run_prep(p, knock={0: np.zeros(hidden_dim())})
    noise = float(max(np.abs(S_noop - S0).max(), np.abs(S_kz - S0).max()))
    assert noise < 0.05, f"zero-steer / zero-knockout changed margins by {noise}"
    assert len(HOOK_REGISTRY) == 0
    # subset consistency (informational noise floor: same rows, different batch composition)
    sub = subset_prep(p, list(range(min(N, 3))))
    drift = float(np.abs(run_prep(sub) - S0[:sub["N"]]).max())
    print(f"  pipeline self-test: no-op noise {noise:.2e}; batch-composition drift {drift:.3f} nats "
          f"(shifts below ~{CONFIG['noise_floor_mult'] * max(drift, noise):.2f} nats are not interpretable)")
    return max(drift, noise)

def other_task_direction(task, src, rec, ctxs, L):
    """Direction from the OTHER task (diff-of-means, at layer L) for the 2x2 specificity check."""
    if task == "gender":
        c = ctxs.get(("order", "DE"))
        if c is None or not c["gate"]["ok"]:
            return None
        return class_direction(get_states(c["D"])[:, L, :], c["D"]["native_sign"])
    c = ctxs.get(("gender", rec["lang"]))
    if c is None or not c["gate"]["ok"]:
        return None
    return class_direction(get_states(c["D"])[:, L, :], c["D"]["native_sign"])

def make_nuis_fn(XS_all, XO_all):
    """dass-vs-ob nuisance direction at ANY layer (v4 only had it at L*)."""
    return lambda L: unit_vec(XS_all[:, L, :].mean(0) - XO_all[:, L, :].mean(0))

def _sigma_at(prep, L):
    return float(np.linalg.norm(get_states(prep)[:, L, :], axis=-1).mean())

def _perm_dirs_at(src, ctxs, L, cfg):
    cache = src.setdefault("perm_cache", {})
    if L not in cache:
        pD = ctxs[(src["task"], src["lang"])]["D"]
        X, sg = get_states(pD)[:, L, :], pD["native_sign"]
        P = permutation_directions(X[sg > 0], X[sg < 0], cfg["n_perm_dirs"], np.random.default_rng(7 + L))
        cache[L] = [unit_vec(v) for v in P]
    return cache[L]

def _src_dose_at(src, L, alphas):
    cache = src.setdefault("dose_cache", {})
    if L not in cache:
        f = src["fits"][L]
        cache[L] = [flip_rate(src["pT"], L, f["u"], a, f["sigma"]) for a in alphas]
    return cache[L]

def cross_cell(src, rec, ctxs, neutral, rand_dirs, rng):
    """RULE-V3 recipient cell. The source's layer-L direction is injected at layer L. L is chosen on the RECIPIENT
    derivation half (max mean margin shift at alpha*) among layers where the source direction is itself sufficient.
    The test half only evaluates. R = recipient test flip / source in-language test flip AT THE SAME LAYER."""
    task, alpha, Ls = src["task"], src["alpha"], src["Lstar"]
    alphas = list(CONFIG["alpha_grid"]) + list(CONFIG["alpha_dose_extra"])
    pRD, pR = rec["D"], rec["T"]
    cell = dict(task=task, source=src["lang"], recipient=rec["lang"], alpha=alpha, Lstar_src=Ls, layer=None,
                R=None, R_ci=(0.0, 0.0), R_ub=None, R_shift=None, flip_recipient=None, controls_ok=False, reason="")
    cell["class"] = "UNUSABLE"
    # ---- 1. recipient layer, derivation half only ----
    sweep = []
    for L in src["elig_layers"]:
        eD = eff_steer(pRD, L, src["fits"][L]["u"], alpha, _sigma_at(pRD, L))
        sweep.append(dict(layer=int(L), shift=mean_or_zero(eD["shift"]), flip=mean_or_zero(eD["flip"])))
    if not sweep:
        cell["reason"] = "no eligible layer (source direction not sufficient at any window layer)"
        return cell
    best = max(sweep, key=lambda d: (d["shift"], d["flip"]))
    L = best["layer"]
    f = src["fits"][L]; u, u0 = f["u"], f["u0"]
    sl = src["layer_eval"][L]
    sigR = _sigma_at(pR, L)
    e = eff_steer(pR, L, u, alpha, sigR)
    n_corr = len(e["flip"])
    # v4-style reference (inject at the SOURCE's L*), so run 4 can be compared directly with run 3
    flip_ref = flip_rate(pR, Ls, src["u_star"], alpha, _sigma_at(pR, Ls))
    cell.update(layer=L, layer_sweep=sweep, deriv_shift_rec=best["shift"], deriv_flip_rec=best["flip"],
                flip_source=sl["flip"], src_ok_at_L=bool(sl["flip"] >= CONFIG["suff_min_flip"]), n_recipient=n_corr,
                suff_ok=src["suff_ok"], necessity_ok=src["nec_ok"], nec_frac=src["nec_frac"], rand_knock_frac=src["rand_knock_frac"],
                flip_rec_at_Lstar=flip_ref, R_at_Lstar_v4style=transfer_ratio(flip_ref, src["flip_src"]))
    if n_corr < CONFIG["min_correct_items"]:
        cell["reason"] = f"recipient has only {n_corr} clean-correct test items"
        return cell
    flip_rec, shift_rec = mean_or_zero(e["flip"]), mean_or_zero(e["shift"])
    R = transfer_ratio(flip_rec, sl["flip"])
    _, lo, hi = bootstrap_ratio_ci(group_by_scene(pR, e, "flip"), sl["groups"], CONFIG["N_BOOT"], rng)
    ub_flip = exact_binom_upper(float(np.sum(e["flip"])), n_corr)
    R_ub = (ub_flip / sl["flip"]) if sl["flip"] > 1e-8 else None
    # ---- 2. controls (same layer L, alpha*, sigma_recipient) -- REPORTED for every cell ----
    rand_mean = float(control_flips(pR, L, alpha, sigR, rand_dirs).mean())
    perm_flips = control_flips(pR, L, alpha, sigR, _perm_dirs_at(src, ctxs, L, CONFIG))
    perm_mean = float(perm_flips.mean())
    cue = unit_vec(neutral[src["lang"]][:, L, :].mean(0) - neutral[rec["lang"]][:, L, :].mean(0))
    cue_flip = flip_rate_both_signs(pR, L, cue, alpha, sigR)
    would_transfer = (R is not None and R >= CONFIG["transfer_min"])
    spec_ok, spec_ratio, nuis_ok, nuis_flip = None, None, None, None
    dm_own = flip_rate(pR, L, u0, alpha, sigR)
    other = other_task_direction(task, src, rec, ctxs, L) if rec["lang"] in BOTH_TASK_LANGS else None
    cross_task_flip = flip_rate_both_signs(pR, L, other, alpha, sigR) if other is not None else None
    if would_transfer and cross_task_flip is not None:
        spec_ok, spec_ratio = specificity_2x2(dm_own, cross_task_flip, CONFIG["specificity_min_ratio"])
    if task == "order" and src.get("nuis_fn") is not None:
        nuis_flip = flip_rate_both_signs(pR, L, src["nuis_fn"](L), alpha, sigR)
        if would_transfer:
            nuis_ok = bool(nuis_flip <= max(CONFIG["nuisance_floor"], 0.5 * flip_rec))
    floor = CONFIG["control_floor_items"] / max(n_corr, 1)
    strongest = max(rand_mean, perm_mean, cue_flip, floor)
    ctrl_ratio = flip_rec / strongest
    controls_ok = bool(ctrl_ratio >= CONFIG["control_ratio_min"] and spec_ok is not False and nuis_ok is not False)
    if task == "gender":
        sgn = pR["native_sign"][e["idx"]]
        cell["flip_fem_native"] = mean_or_zero(e["flip"][sgn > 0]) if (sgn > 0).any() else None
        cell["flip_masc_native"] = mean_or_zero(e["flip"][sgn < 0]) if (sgn < 0).any() else None
    # ---- 3. dose-response (recipient, random-direction control, source in its own language) at layer L ----
    rec_dose = [flip_rate(pR, L, u, a, sigR) for a in alphas]
    rnd_dose = [float(np.mean([flip_rate(pR, L, d, a, sigR) for d in rand_dirs[:CONFIG["n_dose_rand"]]])) for a in alphas]
    src_dose = _src_dose_at(src, L, alphas)
    ed_s, ed_r, ed_n = dose_ed50(alphas, src_dose), dose_ed50(alphas, rec_dose), dose_ed50(alphas, rnd_dose)
    cell["dose"] = [dict(alpha=float(a), flip_rec=float(x), flip_rand=float(y), flip_src=float(z), is_alpha_star=bool(a == alpha))
                    for a, x, y, z in zip(alphas, rec_dose, rnd_dose, src_dose)]
    cell.update(ed50_src=ed_s, ed50_rec=ed_r, ed50_rand=ed_n, sens_ratio=sensitivity_ratio(ed_s, ed_r),
                rec_never_reaches_half=bool(ed_r is None))
    cell.update(flip_recipient=flip_rec, shift_recipient=shift_rec, R=R, R_ci=(lo, hi) if lo is not None else (0.0, 0.0), R_ub=R_ub,
                R_shift=transfer_ratio(shift_rec, sl["shift"]), rand_mean=rand_mean, perm_mean=perm_mean,
                perm_p95=float(np.quantile(perm_flips, 0.95)), cue_flip=cue_flip, dm_own_flip=dm_own,
                cross_task_flip=cross_task_flip, spec_ratio=spec_ratio, spec_ok=spec_ok, nuis_flip=nuis_flip, nuis_ok=nuis_ok,
                controls_ok=controls_ok, ctrl_ratio=ctrl_ratio, ctrl_floor=floor,
                legacy_abs_controls_ok=bool(max(rand_mean, perm_mean, cue_flip) <= 0.10))   # run-3 rule, informational only
    cell["class"] = classify_cell(src["suff_ok"] and cell["src_ok_at_L"], src["nec_ok"], controls_ok, R, cell["R_ci"], CONFIG, R_ub=R_ub)
    if cell["class"] == "UNUSABLE":
        why = []
        if not src["suff_ok"]: why.append(f"source sufficiency failed (in-language flip {src['flip_src']:.2f} < {CONFIG['suff_min_flip']})")
        if not cell["src_ok_at_L"]: why.append(f"source flip at chosen layer {L} is {sl['flip']:.2f} < {CONFIG['suff_min_flip']}")
        if not src["nec_ok"]: why.append(f"source necessity failed (knockout removes {src['nec_frac']:.2f} of margin vs random {src['rand_knock_frac']:.2f})")
        if would_transfer and not controls_ok:
            why.append(f"claims transfer but controls failed (own {flip_rec:.2f} vs strongest control {strongest:.2f} -> ratio {ctrl_ratio:.1f} < {CONFIG['control_ratio_min']}; "
                       f"rand {rand_mean:.2f}, perm {perm_mean:.2f}, cue {cue_flip:.2f}, spec_ok {spec_ok}, nuis_ok {nuis_ok})")
        cell["reason"] = "; ".join(why)
    return cell

def hub_notes(cells):
    notes = []
    for rec in ("FR", "ES"):
        rom = [c for c in cells if c["task"] == "gender" and c["recipient"] == rec and c["source"] in ROMANCE and c["class"] != "UNUSABLE" and c["R"] is not None]
        non = [c for c in cells if c["task"] == "gender" and c["recipient"] == rec and c["source"] not in ROMANCE and c["class"] != "UNUSABLE" and c["R"] is not None]
        if not rom:
            continue
        r_rom = rom[0]["R"]
        if not non:
            notes.append(f"{rec}: HUB_UNTESTED -- no usable non-Romance gender source (AR/HI), so Romance-pair transfer (R={r_rom:.2f}) cannot be separated from FR<->ES proximity.")
        else:
            r_non = float(np.mean([c["R"] for c in non]))
            tag = "HUB_CAVEAT" if (r_rom >= CONFIG["transfer_min"] and r_non < 0.5 * r_rom) else "no hub caveat"
            notes.append(f"{rec}: {tag} -- Romance-source R={r_rom:.2f}, non-Romance sources ({','.join(c['source'] for c in non)}) mean R={r_non:.2f}.")
    return notes

def run_pipeline():
    t0 = time.time()
    rng = np.random.default_rng(2026)
    print("=== 1. build + gate every language/task ===")
    ctxs = {}
    for lang in GENDER_LANGS:
        ctxs[("gender", lang)] = prepare_language("gender", lang)
    for lang in ORDER_LANGS:
        ctxs[("order", lang)] = prepare_language("order", lang)
    for (task, lang), c in ctxs.items():
        g = c["gate"]
        print(f"  {task:6s} {lang}: acc={g['acc']:.2f}  usable deriv/test={g['n_deriv']}/{g['n_test']}  dropped(tokenisation)={g['dropped']}  -> {'PASS' if g['ok'] else 'FAIL'}")
    noise = pipeline_self_tests(next(c for c in ctxs.values() if c["gate"]["ok"] or c["T"]["N"] > 0))

    need_langs = {l for (t, l), c in ctxs.items() if c["gate"]["ok"]}
    neutral = {l: neutral_states(l) for l in need_langs}
    rand_dirs = [unit_vec(rng.normal(size=hidden_dim())) for _ in range(CONFIG["n_random_dirs"])]

    print("\n=== 2. sources: window -> DAS-lite fit -> L*, alpha* -> sufficiency + necessity (in-language) ===")
    sources = {}
    for (task, lang), c in ctxs.items():
        is_source = (task == "gender" and c["gate"]["ok"]) or (task == "order" and lang == "DE" and c["gate"]["ok"])
        if not is_source:
            continue
        s = process_source(c, CONFIG)
        if s["ok"] and task == "order":   # nuisance direction: 'dass' vs 'ob' frames (same order, different complementizer)
            sub = c["D"]; XS = get_states(sub)[sub["native_sign"] < 0][:, s["Lstar"], :]
            XO = get_states(c["OB_D"])[:, s["Lstar"], :] if c["OB_D"]["N"] else None
            s["nuis_dir"] = unit_vec(XS.mean(0) - XO.mean(0)) if XO is not None else None
            s["nuis_fn"] = make_nuis_fn(get_states(sub)[sub["native_sign"] < 0], get_states(c["OB_D"])) if c["OB_D"]["N"] else None
            if s["nuis_dir"] is not None:
                s["nuis_inlang_flip"] = flip_rate_both_signs(c["T"], s["Lstar"], s["nuis_dir"], s["alpha"], s["sigma"])
        sources[(task, lang)] = s
        if not s["ok"]:
            print(f"  {task:6s} {lang}: SOURCE UNUSABLE -- {s['reason']}")
            continue
        nfit = sum(f["used_fit"] for f in s["fits"].values())
        print(f"  {task:6s} {lang}: window={s['window']}  L*={s['Lstar']}  alpha*={s['alpha']}  "
              f"in-lang flip={s['flip_src']:.2f} (n={s['n_src']}) shift={s['shift_src']:.2f} nats  "
              f"SUFF={'ok' if s['suff_ok'] else 'FAIL'}  NEC={'ok' if s['nec_ok'] else 'FAIL'} "
              f"(knockout removes {s['nec_frac']:.2f} of margin; random {s['rand_knock_frac']:.2f})  "
              f"SPSA beat diff-of-means on held-out in {nfit}/{len(s['fits'])} layers; test flip fitted {s['flip_src']:.2f} vs diff-of-means {s['flip_src_u0']:.2f}"
              + ("  *** alpha* AT GRID FLOOR: calibration invalid, extend the grid ***" if s["alpha_at_floor"] else ""))
        print(f"           eligible injection layers (source sufficient on derivation at alpha*): {s['elig_layers']}")

    print("\n=== 3. recipients: cross-language transfer + controls ===")
    cells = []
    for (task, src_lang), s in sources.items():
        if not s["ok"]:
            continue
        for (t2, rec_lang), rc in ctxs.items():
            if t2 != task or rec_lang == src_lang or not rc["gate"]["ok"]:
                continue
            if task == "order" and rec_lang == "DE":
                continue
            cell = cross_cell(s, rc, ctxs, neutral, rand_dirs, rng)
            cells.append(cell)
            r = "n/a" if cell["R"] is None else f"{cell['R']:.2f} [{cell['R_ci'][0]:.2f},{cell['R_ci'][1]:.2f}] ub={cell['R_ub'] if cell['R_ub'] is None else round(cell['R_ub'], 2)}"
            v4 = cell.get("R_at_Lstar_v4style")
            sens = cell.get("sens_ratio")
            print(f"  {task:6s} {src_lang}->{rec_lang}: L={cell.get('layer')} (src L*={cell['Lstar_src']})  R={r}  flip_rec={cell['flip_recipient'] if cell['flip_recipient'] is None else round(cell['flip_recipient'], 2)}  "
                  f"[run3-style R@L*={'n/a' if v4 is None else round(v4, 2)}; ED50 ratio={'n/a' if sens is None else round(sens, 2)}]  -> {cell['class']}  {cell['reason']}")
            if cell.get("ctrl_ratio") is not None:
                print(f"           controls: rand {cell['rand_mean']:.2f} perm {cell['perm_mean']:.2f} (p95 {cell['perm_p95']:.2f}) cue {cell['cue_flip']:.2f} -> ratio {cell['ctrl_ratio']:.1f}"
                      f"  (legacy abs-0.10 rule would say {'ok' if cell['legacy_abs_controls_ok'] else 'FAIL'})")

    print("\n=== 4. verdict (primary cells only: gender ES->FR, FR->ES; order DE->FR, DE->ES) ===")
    primary = []
    for task, pairs in PRIMARY.items():
        for src_lang, rec_lang in pairs:
            m = [c for c in cells if c["task"] == task and c["source"] == src_lang and c["recipient"] == rec_lang]
            if m:
                primary.append(m[0])
            else:
                why = "source or recipient failed the clean-accuracy gate / produced no usable source"
                primary.append(dict(task=task, source=src_lang, recipient=rec_lang, **{"class": "UNUSABLE"}, R=None, R_ci=(0.0, 0.0), reason=why))
    v = verdict(primary, CONFIG)
    hubs = hub_notes(cells)
    print(v)
    for h in hubs:
        print("  hub check:", h)
    print(f"  (instrument noise floor ~{noise:.3f} nats; total runtime {time.time() - t0:.0f}s)")

    # ------------------------------ outputs ------------------------------
    tag = os.path.basename(str(getattr(getattr(model, "config", None), "_name_or_path", "") or "toy")) or "toy"
    def _clean(o):
        if isinstance(o, (np.floating, np.integer)): return o.item()
        if isinstance(o, (np.ndarray,)): return o.tolist()
        if isinstance(o, tuple): return list(o)
        return str(o)
    cols = ["task", "source", "recipient", "layer", "Lstar_src", "alpha", "n_recipient", "flip_source", "src_ok_at_L", "flip_recipient", "R", "R_ci", "R_ub", "R_shift",
            "class", "reason", "suff_ok", "necessity_ok", "nec_frac", "rand_knock_frac", "controls_ok", "ctrl_ratio", "ctrl_floor", "rand_mean", "perm_mean",
            "perm_p95", "cue_flip", "dm_own_flip", "cross_task_flip", "spec_ratio", "spec_ok", "nuis_flip", "nuis_ok", "legacy_abs_controls_ok",
            "flip_fem_native", "flip_masc_native", "deriv_shift_rec", "deriv_flip_rec", "flip_rec_at_Lstar", "R_at_Lstar_v4style",
            "ed50_src", "ed50_rec", "ed50_rand", "sens_ratio", "rec_never_reaches_half"]
    with open(os.path.join(RESULTS_DIR, f"exp29_cell_report_{tag}.csv"), "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(cols)
        for c in cells:
            w.writerow([json.dumps(c.get(k), default=_clean) if k == "R_ci" else c.get(k) for k in cols])
    with open(os.path.join(RESULTS_DIR, f"exp29_layer_curves_{tag}.csv"), "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(["task", "lang", "layer", "loo_shift_nats", "loo_flip", "in_window", "is_Lstar"])
        for (task, lang), s in sources.items():
            for cv in s["curve"]:
                w.writerow([task, lang, cv["layer"], cv["shift"], cv["flip"], cv["layer"] in s["window"], cv["layer"] == s.get("Lstar")])
    with open(os.path.join(RESULTS_DIR, f"exp29_dose_response_{tag}.csv"), "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(["task", "source", "recipient", "layer", "alpha", "flip_recipient", "flip_random_ctrl", "flip_source_same_layer", "is_alpha_star"])
        for (task, lang), sx in sources.items():
            for d in sx.get("dose_self", []):
                w.writerow([task, lang, "SELF(test)", sx.get("Lstar"), d["alpha"], d["flip"], "", d["flip"], d["alpha"] == sx["alpha"]])
        for c in cells:
            for d in c.get("dose", []):
                w.writerow([c["task"], c["source"], c["recipient"], c.get("layer"), d["alpha"], d["flip_rec"], d["flip_rand"], d["flip_src"], d["is_alpha_star"]])
    with open(os.path.join(RESULTS_DIR, f"exp29_recipient_layer_sweep_{tag}.csv"), "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(["task", "source", "recipient", "layer", "deriv_shift_nats", "deriv_flip", "chosen"])
        for c in cells:
            for d in c.get("layer_sweep", []):
                w.writerow([c["task"], c["source"], c["recipient"], d["layer"], d["shift"], d["flip"], d["layer"] == c.get("layer")])
    src_summary = {f"{t}:{l}": {k: s.get(k) for k in ["ok", "reason", "window", "Lstar", "alpha", "flip_src", "shift_src", "n_src", "suff_ok",
                    "nec_ok", "nec_frac", "nec_flip", "rand_knock_frac", "nuis_inlang_flip", "flip_src_u0", "alpha_at_floor", "elig_layers"]}
                   | {"spsa_used_per_layer": {str(L): bool(f["used_fit"]) for L, f in s.get("fits", {}).items()},
                      "layer_eval": {str(L): dict(flip=v["flip"], shift=v["shift"], n=v["n"]) for L, v in s.get("layer_eval", {}).items()}}
                   for (t, l), s in sources.items()}
    json.dump(dict(verdict=v, hub_notes=hubs, gates={f"{t}:{l}": c["gate"] for (t, l), c in ctxs.items()}, sources=src_summary,
                   noise_floor_nats=noise, config=CONFIG), open(os.path.join(RESULTS_DIR, f"verdict_{tag}.json"), "w"), indent=2, default=_clean)
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(8, 4))
        for (task, lang), s in sources.items():
            ax.plot([c["layer"] for c in s["curve"]], [c["shift"] for c in s["curve"]], marker="o", ms=3, label=f"{task}:{lang}")
        ax.set_xlabel("layer"); ax.set_ylabel("leave-one-out diff-of-means shift (nats)"); ax.legend(fontsize=7)
        fig.savefig(os.path.join(RESULTS_DIR, f"exp29_loo_curves_{tag}.png"), dpi=130, bbox_inches="tight"); plt.close(fig)
    except Exception as ex:
        print("  (plot skipped:", ex, ")")
    print(f"\nwrote CSV/JSON/PNG to {RESULTS_DIR}/ (tag={tag})")
    return dict(ctxs=ctxs, sources=sources, cells=cells, verdict=v, hubs=hubs)

RUN_READY = globals().get("GENDER_BANK_READY", False) and globals().get("ORDER_BANK_READY", False)
if not RUN_READY:
    print("Skipping the full run: an item bank failed its count/balance check. See Cells 4-5.")
else:
    RESULTS = run_pipeline()


=== 1. build + gate every language/task ===
  gender FR: acc=1.00  usable deriv/test=20/20  dropped(tokenisation)=0  -> PASS
  gender ES: acc=1.00  usable deriv/test=20/20  dropped(tokenisation)=0  -> PASS
  gender HI: acc=0.97  usable deriv/test=20/20  dropped(tokenisation)=0  -> PASS
  gender AR: acc=1.00  usable deriv/test=20/20  dropped(tokenisation)=0  -> PASS
  order  EN: acc=1.00  usable deriv/test=12/12  dropped(tokenisation)=0  -> PASS
  order  FR: acc=1.00  usable deriv/test=12/12  dropped(tokenisation)=0  -> PASS
  order  ES: acc=1.00  usable deriv/test=12/12  dropped(tokenisation)=0  -> PASS
  order  DE: acc=0.98  usable deriv/test=24/24  dropped(tokenisation)=0  -> PASS
  order  HI: acc=0.88  usable deriv/test=12/12  dropped(tokenisation)=0  -> PASS
  order  JA: acc=1.00  usable deriv/test=11/11  dropped(tokenisation)=2  -> PASS
  order  KO: acc=1.00  usable deriv/test=12/12  dropped(tokenisation)=0  -> PASS
  order  TR: acc=1.00  usable deriv/test=12/12  dropped(tokenisat

In [16]:
def _mk_cell(task, recipient, cls, R, ci):
    return dict(task=task, recipient=recipient, class_=cls, R=R, R_ci=ci, **{"class": cls})

def force_verdict_branches(cfg):
    def cells(gender_classes, gender_Rs, order_classes, order_Rs):
        out = []
        for recip, c, r in zip(("FR", "ES"), gender_classes, gender_Rs):
            out.append(dict(task="gender", recipient=recip, **{"class": c}, R=r, R_ci=(max(0,r-0.1), r+0.1) if r is not None else (0,0)))
        for recip, c, r in zip(("FR", "ES"), order_classes, order_Rs):
            out.append(dict(task="order", recipient=recip, **{"class": c}, R=r, R_ci=(max(0,r-0.1), r+0.1) if r is not None else (0,0)))
        return out

    v1 = verdict(cells(["TRANSFER","TRANSFER"], [0.8,0.75], ["WALL","WALL"], [0.05,0.1]), cfg)
    v2 = verdict(cells(["TRANSFER","TRANSFER"], [0.8,0.75], ["TRANSFER","TRANSFER"], [0.7,0.8]), cfg)
    v3 = verdict(cells(["WALL","WALL"], [0.05,0.1], ["TRANSFER","TRANSFER"], [0.7,0.8]), cfg)
    v4 = verdict(cells(["TRANSFER","UNUSABLE"], [0.8,None], ["WALL","WALL"], [0.05,0.1]), cfg)
    for name, v in [("inversion",v1),("both_shared",v2),("reversed",v3),("inadequate",v4)]:
        print(name, "->", v[:70])
    assert v1.startswith("INVERSION_SUPPORTED")
    assert v2.startswith("BOTH_SHARED")
    assert v3.startswith("REVERSED")
    assert v4.startswith("INSTRUMENT_INADEQUATE")
    print("VERDICT BRANCHES OK")

force_verdict_branches(CONFIG)

# ---- direct unit tests on the smaller helper functions (not exercised above) ----
def test_classify_cell(cfg):
    assert classify_cell(True, True, True, 0.8, (0.6, 1.0), cfg) == "TRANSFER"
    assert classify_cell(True, True, True, 0.05, (0.0, 0.15), cfg) == "WALL"
    assert classify_cell(True, True, True, 0.35, (0.2, 0.5), cfg) == "AMBIGUOUS"
    assert classify_cell(False, True, True, 0.8, (0.6, 1.0), cfg) == "UNUSABLE"   # sufficiency gate failed
    assert classify_cell(True, False, True, 0.8, (0.6, 1.0), cfg) == "UNUSABLE"   # necessity gate failed
    assert classify_cell(True, True, False, 0.8, (0.6, 1.0), cfg) == "UNUSABLE"   # claims transfer, controls failed
    assert classify_cell(True, True, False, 0.05, (0.0, 0.15), cfg) == "WALL"      # rule-v3: controls gate ONLY claimed transfer
    assert classify_cell(True, True, False, 0.35, (0.2, 0.5), cfg) == "AMBIGUOUS"
    assert classify_cell(True, True, True, 0.0, (0.0, 0.0), cfg, R_ub=0.9) == "WALL_LOWPOWER"   # 0/12 vs a 0.78 source
    assert classify_cell(True, True, True, 0.0, (0.0, 0.0), cfg, R_ub=0.34) == "WALL"
    print("classify_cell OK")

def test_v5_helpers():
    assert abs(exact_binom_upper(0, 12) - 0.2646) < 5e-3, exact_binom_upper(0, 12)
    assert exact_binom_upper(12, 12) == 1.0 and exact_binom_upper(0, 0) == 1.0
    assert abs(exact_binom_upper(3, 20) - 0.3789) < 5e-3, exact_binom_upper(3, 20)
    assert abs(dose_ed50([0.1, 0.2, 0.5], [0.1, 0.4, 0.9]) - 0.2 * 2.5 ** 0.2) < 1e-6
    assert dose_ed50([0.1, 0.2], [0.0, 0.1]) is None and dose_ed50([0.1, 0.2], [0.7, 0.9]) == 0.1
    assert sensitivity_ratio(0.2, None) == 0.0 and sensitivity_ratio(None, 0.3) is None and abs(sensitivity_ratio(0.2, 0.1) - 2.0) < 1e-9
    print("exact_binom_upper / dose_ed50 / sensitivity_ratio OK")

test_v5_helpers()

def test_bootstrap_ci():
    const_vals = [np.array([3.0]) for _ in range(20)]
    mean, lo, hi = bootstrap_ci(const_vals, lambda sample: float(np.mean([s[0] for s in sample])), n_boot=200)
    assert abs(mean - 3.0) < 1e-6 and abs(lo - 3.0) < 1e-6 and abs(hi - 3.0) < 1e-6, (mean, lo, hi)
    print("bootstrap_ci OK (degenerate zero-width case)")

def test_random_and_permutation_directions():
    V = random_directions(d=16, n=50)
    assert V.shape == (50, 16)
    norms = np.linalg.norm(V, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-6)
    rng = np.random.default_rng(0)
    a = rng.normal(size=(10, 16)) + 2.0
    b = rng.normal(size=(10, 16)) - 2.0
    P = permutation_directions(a, b, n=30)
    assert P.shape == (30, 16)
    print("random_directions / permutation_directions OK")

def test_cue_only_and_specificity():
    rng = np.random.default_rng(0)
    x = rng.normal(size=(10, 8)) + 1.0
    y = rng.normal(size=(10, 8)) - 1.0
    cue = cue_only_direction(x, y)
    assert cue.shape == (8,)
    ok, ratio = specificity_2x2(effect_own_task=0.6, effect_cross_task=0.1)
    assert ok and abs(ratio - 6.0) < 1e-9, ratio
    ok2, ratio2 = specificity_2x2(effect_own_task=0.6, effect_cross_task=0.3)
    assert not ok2 and abs(ratio2 - 2.0) < 1e-9
    print("cue_only_direction / specificity_2x2 OK")

def test_transfer_ratio_edge_case():
    assert transfer_ratio(0.4, 0.8) == 0.5
    assert transfer_ratio(0.4, 0.0) is None  # source gate should already have failed upstream
    print("transfer_ratio OK")

test_classify_cell(CONFIG)
test_bootstrap_ci()
test_random_and_permutation_directions()
test_cue_only_and_specificity()
test_transfer_ratio_edge_case()
print("ALL HELPER UNIT TESTS PASSED")


inversion -> INVERSION_SUPPORTED: gender transfers, order does not, CIs separate, c
both_shared -> BOTH_SHARED: both gender and order transfer -- not the inversion.
reversed -> REVERSED: order transfers as much as or more than gender, CIs separate
inadequate -> INSTRUMENT_INADEQUATE: at least one required FR/ES cell is UNUSABLE (g
VERDICT BRANCHES OK
exact_binom_upper / dose_ed50 / sensitivity_ratio OK
classify_cell OK
bootstrap_ci OK (degenerate zero-width case)
random_directions / permutation_directions OK
cue_only_direction / specificity_2x2 OK
transfer_ratio OK
ALL HELPER UNIT TESTS PASSED


In [17]:
import os, math
import numpy as np, pandas as pd

D, TAG = "exp29_results_run4", "gemma-2-9b-it"
cells = pd.read_csv(f"{D}/exp29_cell_report_{TAG}.csv")
dose  = pd.read_csv(f"{D}/exp29_dose_response_{TAG}.csv")
sweep = pd.read_csv(f"{D}/exp29_recipient_layer_sweep_{TAG}.csv")
pd.set_option("display.width", 200, "display.max_columns", 30)

# ---------- 1. Does "never reaches 0.5" mean anything? random-direction curve vs recipient curve (order) ----------
d = dose[(dose.task == "order") & (dose.recipient != "SELF(test)")].copy()
d["excess_over_random"] = d.flip_recipient - d.flip_random_ctrl
print("=== ORDER: recipient flip / random-direction flip by alpha ===")
for col in ["flip_recipient", "flip_random_ctrl", "excess_over_random"]:
    print(f"\n-- {col}")
    print(d.pivot_table(index="alpha", columns="recipient", values=col).round(2))

print("\n=== ED50 summary (order cells; NaN = never reaches 0.5 on the alpha grid up to 4) ===")
print(cells[cells.task == "order"][["source", "recipient", "layer", "n_recipient", "flip_source", "flip_recipient",
      "R", "R_ub", "ed50_src", "ed50_rec", "ed50_rand", "sens_ratio", "cue_flip", "class"]].round(2).to_string(index=False))
# If ed50_rand is close to ed50_src, large alphas are non-specific, so a recipient that "never reaches 0.5" is weaker evidence.

# ---------- 2. Is DE->EN (R=0.43) a spike at one layer or broad? (RECIPIENT DERIVATION half, from the layer sweep) ----------
s = sweep[(sweep.task == "order") & (sweep.source == "DE")]
print("\n=== layer sweep, derivation half: mean margin shift (nats) by layer x recipient ===")
print(s.pivot_table(index="layer", columns="recipient", values="deriv_shift_nats").round(2))
print("\n=== same, derivation flip rate (1 item = 0.083 at n=12) ===")
print(s.pivot_table(index="layer", columns="recipient", values="deriv_flip").round(2))
en = s[s.recipient == "EN"].sort_values("layer")
print(f"\nEN: {int((en.deriv_flip >= 0.25).sum())}/{len(en)} layers with derivation flip >= 0.25; "
      f"chosen layer = {int(en[en.chosen].layer.iloc[0])}; shift range {en.deriv_shift_nats.min():.2f}..{en.deriv_shift_nats.max():.2f}")

# ---------- 3. How many items for a real wall? exact upper bound of R after 0 (or 1) flips ----------
def cp_upper(k, n, alpha=0.05):
    if n <= 0 or k >= n: return 1.0
    lo, hi = 0.0, 1.0
    for _ in range(60):
        mid = (lo + hi) / 2
        cdf = sum(math.comb(n, j) * mid**j * (1 - mid)**(n - j) for j in range(int(k) + 1))
        lo, hi = (mid, hi) if cdf > alpha / 2 else (lo, mid)
    return hi

print("\n=== recipient items needed per half so that R_ub <= target (k = flips observed) ===")
rows = []
for src_flip in (0.78, 0.85, 0.90):
    for k in (0, 1):
        row = {"source_flip": src_flip, "k": k}
        for target in (0.5, 0.3, 0.2):
            n = next((n for n in range(max(k + 1, 4), 400) if cp_upper(k, n) / src_flip <= target), None)
            row[f"R_ub<={target}"] = n
        rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

=== ORDER: recipient flip / random-direction flip by alpha ===

-- flip_recipient
recipient    EN    ES    FR   HI   JA   KO    TR
alpha                                           
0.02       0.00  0.00  0.00  0.0  0.0  0.0  0.00
0.05       0.00  0.00  0.00  0.0  0.0  0.0  0.00
0.10       0.00  0.00  0.00  0.0  0.0  0.0  0.00
0.20       0.00  0.00  0.00  0.0  0.0  0.0  0.00
0.35       0.33  0.08  0.08  0.0  0.0  0.0  0.08
0.50       0.50  0.25  0.08  0.0  0.0  0.0  0.17
1.00       0.75  0.50  0.17  0.1  0.0  0.0  0.08
2.00       0.92  0.58  0.42  0.1  0.0  0.0  0.08
3.00       1.00  0.33  0.42  0.1  0.0  0.0  0.17
4.00       1.00  0.25  0.42  0.1  0.0  0.0  0.17

-- flip_random_ctrl
recipient    EN    ES    FR    HI    JA    KO    TR
alpha                                              
0.02       0.00  0.00  0.00  0.00  0.00  0.00  0.00
0.05       0.00  0.00  0.00  0.00  0.00  0.00  0.00
0.10       0.00  0.00  0.00  0.01  0.00  0.00  0.00
0.20       0.00  0.00  0.00  0.04  0.00  0.00  0.

In [18]:
def layer_profile(recipient, task="order", source="DE", n_rand=20):
    src = RESULTS["sources"][(task, source)]
    pR  = RESULTS["ctxs"][(task, recipient)]["T"]
    alpha = src["alpha"]
    rng = np.random.default_rng(0)
    rdirs = [unit_vec(rng.normal(size=hidden_dim())) for _ in range(n_rand)]
    out = []
    for L in src["elig_layers"]:
        f, sl = src["fits"][L], src["layer_eval"][L]
        sigR = _sigma_at(pR, L)
        e = eff_steer(pR, L, f["u"], alpha, sigR)
        k, n = int(np.sum(e["flip"])), len(e["flip"])
        rnd = float(np.mean([flip_rate(pR, L, d, alpha, sigR) for d in rdirs]))
        out.append(dict(layer=L, n=n, k=k, rec_flip=k / max(n, 1), rand_flip=rnd,
                        src_flip=sl["flip"], R=(k / max(n, 1)) / sl["flip"] if sl["flip"] > 1e-8 else np.nan,
                        R_ub=exact_binom_upper(k, n) / sl["flip"] if sl["flip"] > 1e-8 else np.nan,
                        shift=mean_or_zero(e["shift"])))
    return pd.DataFrame(out).round(2)

for rcp in ["EN", "FR", "ES"]:
    print(f"\n=== DE -> {rcp}: test-half profile over DE's eligible layers (alpha*={RESULTS['sources'][('order','DE')]['alpha']}) ===")
    print(layer_profile(rcp).to_string(index=False))


=== DE -> EN: test-half profile over DE's eligible layers (alpha*=0.35) ===
 layer  n  k  rec_flip  rand_flip  src_flip    R  R_ub  shift
    24 12  0      0.00        0.0      0.74 0.00  0.36   2.38
    26 12  0      0.00        0.0      0.70 0.00  0.38   4.82
    28 12  1      0.08        0.0      0.78 0.11  0.49   6.61
    30 12  2      0.17        0.0      0.83 0.20  0.59   9.21
    32 12  3      0.25        0.0      0.83 0.30  0.69  10.43
    34 12  4      0.33        0.0      0.78 0.43  0.83  10.49
    36 12  4      0.33        0.0      0.78 0.43  0.83  10.42
    38 12  3      0.25        0.0      0.70 0.36  0.82   9.16
    40 12  0      0.00        0.0      0.65 0.00  0.41   3.92

=== DE -> FR: test-half profile over DE's eligible layers (alpha*=0.35) ===
 layer  n  k  rec_flip  rand_flip  src_flip    R  R_ub  shift
    24 12  0      0.00        0.0      0.74 0.00  0.36   0.78
    26 12  0      0.00        0.0      0.70 0.00  0.38   2.63
    28 12  0      0.00        0.0      0

In [19]:
c = pd.read_csv(f"{D}/exp29_cell_report_{TAG}.csv")
print(c[["task","source","recipient","layer","R","R_shift","sens_ratio","class"]].round(2).to_string(index=False))

  task source recipient  layer    R  R_shift  sens_ratio     class
gender     FR        ES     34 1.06     0.98        1.02  TRANSFER
gender     FR        HI     28 0.56     0.90        0.66  TRANSFER
gender     FR        AR     28 0.94     1.11        0.97  TRANSFER
gender     ES        FR     32 0.83     0.75        0.92  TRANSFER
gender     ES        HI     32 0.35     0.77        0.59 AMBIGUOUS
gender     ES        AR     32 0.56     0.90        0.69  TRANSFER
gender     HI        FR     34 1.06     1.14        1.85  TRANSFER
gender     HI        ES     34 1.06     1.03        1.35  TRANSFER
gender     HI        AR     32 1.00     1.13        1.16  TRANSFER
gender     AR        FR     34 1.06     0.99        1.17  TRANSFER
gender     AR        ES     34 1.25     1.01        1.17  TRANSFER
gender     AR        HI     34 0.86     0.87        0.83  TRANSFER
 order     DE        EN     36 0.43     1.13        0.47 AMBIGUOUS
 order     DE        FR     34 0.11     0.63        0.00      

In [20]:
import numpy as np, pandas as pd

def shift_controls(task="order", source="DE", n_rand=30, seed=0):
    src = RESULTS["sources"][(task, source)]
    rng = np.random.default_rng(seed)
    rdirs = [unit_vec(rng.normal(size=hidden_dim())) for _ in range(n_rand)]
    rows = []
    for c in RESULTS["cells"]:
        if c["task"] != task or c["source"] != source:
            continue
        rec, L, a = c["recipient"], c["layer"], c["alpha"]
        pR = RESULTS["ctxs"][(task, rec)]["T"]
        sig = _sigma_at(pR, L)
        u = src["fits"][L]["u"]
        e_dir = eff_steer(pR, L, u, a, sig)          # push AWAY from native class
        e_neg = eff_steer(pR, L, -u, a, sig)         # opposite push: strengthens native class (shift < 0)
        rs = np.array([mean_or_zero(eff_steer(pR, L, d, a, sig)["shift"]) for d in rdirs])
        rows.append(dict(
            recipient=rec, layer=L, n=len(e_dir["shift"]),
            clean_margin=float(np.mean(e_dir["m0"])),
            shift_dir=mean_or_zero(e_dir["shift"]),
            shift_opposite=mean_or_zero(e_neg["shift"]),
            shift_rand_mean=float(rs.mean()),
            shift_rand_p95=float(np.quantile(rs, 0.95)),
            shift_rand_absmax=float(np.abs(rs).max()),
            src_shift=src["layer_eval"][L]["shift"],
        ))
    df = pd.DataFrame(rows)
    df["R_shift"] = df.shift_dir / df.src_shift
    df["shift_over_rand_p95"] = df.shift_dir / df.shift_rand_p95.clip(lower=0.05)
    return df.round(2)

print(shift_controls("order", "DE").to_string(index=False))

recipient  layer  n  clean_margin  shift_dir  shift_opposite  shift_rand_mean  shift_rand_p95  shift_rand_absmax  src_shift  R_shift  shift_over_rand_p95
       EN     36 12         11.72      10.42           -4.17             0.53            1.87               2.02       9.19     1.13                 5.58
       FR     34 12         12.82       5.79           -3.95             0.33            1.20               1.48       9.17     0.63                 4.84
       ES     36 12         11.34       6.96           -3.99             0.58            1.38               1.50       9.19     0.76                 5.05
       HI     30 10          3.86       0.10           -0.64            -0.07            0.21               0.89       9.77     0.01                 0.48
       JA     30 11         13.77       0.76            0.17            -0.02            0.28               0.47       9.77     0.08                 2.73
       KO     30 12         13.16       0.62            0.03             0.0

In [21]:
import numpy as np, pandas as pd

def cross_specificity():
    src  = RESULTS["sources"][("order", "DE")]
    a    = src["alpha"]
    gctx = RESULTS["ctxs"][("gender", "FR")]
    rows = []
    for c in RESULTS["cells"]:
        if c["task"] != "order":
            continue
        rec, L = c["recipient"], c["layer"]
        pR  = RESULTS["ctxs"][("order", rec)]["T"]
        sig = _sigma_at(pR, L)
        dirs = {"german_order": src["fits"][L]["u"]}
        if src.get("nuis_fn") is not None:
            dirs["dass_vs_ob"] = src["nuis_fn"](L)
        gD = gctx["D"]
        dirs["gender_FR"] = class_direction(get_states(gD)[:, L, :], gD["native_sign"])
        r = dict(recipient=rec, layer=L)
        for name, u in dirs.items():
            r[name] = mean_or_zero(eff_steer(pR, L, u, a, sig)["shift"])
            r[name + "_opp"] = mean_or_zero(eff_steer(pR, L, -u, a, sig)["shift"])
        rows.append(r)
    return pd.DataFrame(rows).round(2)

def german_on_gender(layers=(30, 34, 36)):
    src = RESULTS["sources"][("order", "DE")]
    a = src["alpha"]
    rows = []
    for rec in ["FR", "ES", "HI", "AR"]:
        pT = RESULTS["ctxs"][("gender", rec)]["T"]
        for L in layers:
            if L not in src["fits"]:
                continue
            sig = _sigma_at(pT, L)
            u = src["fits"][L]["u"]
            e1, e2 = eff_steer(pT, L, u, a, sig), eff_steer(pT, L, -u, a, sig)
            rows.append(dict(recipient=rec, layer=L, clean_margin=float(np.mean(e1["m0"])),
                             shift=mean_or_zero(e1["shift"]), shift_opp=mean_or_zero(e2["shift"])))
    return pd.DataFrame(rows).round(2)

print(cross_specificity().to_string(index=False))
print(german_on_gender().to_string(index=False))

recipient  layer  german_order  german_order_opp  dass_vs_ob  dass_vs_ob_opp  gender_FR  gender_FR_opp
       EN     36         10.42             -4.17       -1.11            2.05       1.19          -0.27
       FR     34          5.79             -3.95       -0.04            1.45       0.84          -0.47
       ES     36          6.96             -3.99       -0.54            1.58       1.04           0.06
       HI     30          0.10             -0.64       -0.16            0.40      -0.03           0.04
       JA     30          0.76              0.17       -0.07            0.18       0.18          -0.33
       KO     30          0.62              0.03        0.07           -0.04       0.02           0.11
       TR     30         -0.18              0.13        0.25           -0.36      -0.04           0.49
recipient  layer  clean_margin  shift  shift_opp
       FR     30          3.33   0.35      -0.02
       FR     34          3.33   0.89      -0.12
       FR     36          3.3

In [22]:
# ===================== ZH recipient test for the German order direction (run AFTER RESULTS = run_pipeline()) =====================
# PRE-REGISTERED PREDICTION (written before running; this test is post-hoc relative to run 4, so log it as such):
#   Hypothesis H_VO ("order direction is shared within head-first languages"): ZH is verb-medial, non-Latin, far from German.
#       -> R_shift >= 0.5 AND shift >= 3x its random-direction p95 AND opposite-sign shift negative (EN/FR/ES: R_shift 0.6-1.1).
#   Hypothesis H_dist ("transfer decays with distance from German / script"): ZH behaves like HI/JA/KO/TR.
#       -> R_shift < 0.1 and shift within the random p95 (HI/JA/KO/TR: R_shift -0.02..0.08).
#   Anything in between = uninformative at n=12; do not interpret.
# CAVEATS: my own ZH items, NO native-speaker review. Mandarin partly licenses O-before-V (topic-fronting), so the OV
# alternative is less clearly "wrong" than in EN/FR/ES: check clean_margin and the gate before reading anything.
import numpy as np, pandas as pd

ZH_SCENES = [  # (subject prefix, VO continuation, OV continuation); same 24 scenes as ORDER_SCENES, verbs chosen so the OV string is not an intransitive reading
 ("男人",   "看书。",     "书看。"),     ("女人",   "推门。",     "门推。"),     ("男孩",   "吃苹果。",   "苹果吃。"),
 ("女孩",   "喝水。",     "水喝。"),     ("老师",   "写信。",     "信写。"),     ("孩子",   "提袋子。",   "袋子提。"),
 ("厨师",   "做汤。",     "汤做。"),     ("司机",   "洗车。",     "车洗。"),     ("医生",   "看报告。",   "报告看。"),
 ("农夫",   "买面包。",   "面包买。"),   ("学生",   "做作业。",   "作业做。"),   ("面包师", "烤蛋糕。",   "蛋糕烤。"),
 ("歌手",   "唱歌曲。",   "歌曲唱。"),   ("爸爸",   "喝茶。",     "茶喝。"),     ("妈妈",   "煮米饭。",   "米饭煮。"),
 ("球员",   "踢球。",     "球踢。"),     ("护士",   "拿药。",     "药拿。"),     ("画家",   "画房子。",   "房子画。"),
 ("工人",   "修墙。",     "墙修。"),     ("奶奶",   "织毛衣。",   "毛衣织。"),   ("飞行员", "查地图。",   "地图查。"),
 ("保安",   "看大门。",   "大门看。"),   ("邻居",   "修篱笆。",   "篱笆修。"),   ("姐姐",   "发消息。",   "消息发。"),
]
assert len(ZH_SCENES) == len(ORDER_SCENES) == 24

JOIN["ZH"] = ""                                  # Chinese is written without spaces (same treatment as JA)
ORDER_NATIVE["ZH"] = "head_first"
NEUTRAL["ZH"] = ["今天天气", "我的名字是", "会议开始于", "我住在", "请关上"]
if not any(u["lang"] == "ZH" for u in CLAUSE_BANK):
    for si, (pre, vo, ov) in enumerate(ZH_SCENES):
        CLAUSE_BANK.append(dict(lang="ZH", scene=si, prefix=pre, cand_vo=vo, cand_ov=ov, native="head_first", split=_split_of(si)))

# ---- 1. gate (the same gate every other language had to pass) ----
ctx_zh = prepare_language("order", "ZH")
g = ctx_zh["gate"]
print(f"order ZH: acc={g['acc']:.2f}  usable deriv/test={g['n_deriv']}/{g['n_test']}  dropped(tokenisation)={g['dropped']}  -> {'PASS' if g['ok'] else 'FAIL'}")
S0 = np.concatenate([ensure_clean(ctx_zh["D"]) * ctx_zh["D"]["native_sign"], ensure_clean(ctx_zh["T"]) * ctx_zh["T"]["native_sign"]])
print(f"  native margin (nats): mean {S0.mean():.2f}, median {np.median(S0):.2f}, share of items with margin<1: {(S0 < 1).mean():.2f}")
if not g["ok"]:
    raise SystemExit("ZH failed the clean-accuracy gate: do not interpret any transfer number for it.")

# ---- 2. the notebook's own cross-cell (layer chosen on ZH DERIVATION half, controls, dose curve) ----
src = RESULTS["sources"][("order", "DE")]
rng = np.random.default_rng(2027)
neutral = {l: neutral_states(l) for l in ("DE", "ZH")}
rand_dirs = [unit_vec(rng.normal(size=hidden_dim())) for _ in range(CONFIG["n_random_dirs"])]
ctxs = dict(RESULTS["ctxs"]); ctxs[("order", "ZH")] = ctx_zh
cell = cross_cell(src, ctx_zh, ctxs, neutral, rand_dirs, rng)
RESULTS["ctxs"][("order", "ZH")] = ctx_zh
RESULTS["cells"] = [c for c in RESULTS["cells"] if not (c["task"] == "order" and c["recipient"] == "ZH")] + [cell]

r = "n/a" if cell["R"] is None else f"{cell['R']:.2f} [{cell['R_ci'][0]:.2f},{cell['R_ci'][1]:.2f}] ub={cell['R_ub']:.2f}"
print(f"\norder DE->ZH: L={cell['layer']}  n={cell['n_recipient']}  R(flip)={r}  flip_rec={cell['flip_recipient']}  R_shift={cell['R_shift']:.2f}  "
      f"cue={cell['cue_flip']:.2f}  ED50 ratio={cell['sens_ratio']}  -> {cell['class']} {cell['reason']}")

# ---- 3. the readout that matters: margin shift vs random-direction p95, both signs (same code as the EN/FR/ES/TR table) ----
def zh_shift_controls(n_rand=30, seed=0):
    rr = np.random.default_rng(seed)
    rdirs = [unit_vec(rr.normal(size=hidden_dim())) for _ in range(n_rand)]
    rows = []
    for rec in ["EN", "FR", "ES", "HI", "JA", "KO", "TR", "ZH"]:
        c = next(x for x in RESULTS["cells"] if x["task"] == "order" and x["recipient"] == rec)
        L, a = c["layer"], c["alpha"]
        pR = RESULTS["ctxs"][("order", rec)]["T"]; sig = _sigma_at(pR, L); u = src["fits"][L]["u"]
        e1, e2 = eff_steer(pR, L, u, a, sig), eff_steer(pR, L, -u, a, sig)
        rs = np.array([mean_or_zero(eff_steer(pR, L, d, a, sig)["shift"]) for d in rdirs])
        p95 = float(np.quantile(rs, 0.95))
        rows.append(dict(recipient=rec, layer=L, n=len(e1["shift"]), clean_margin=float(np.mean(e1["m0"])), shift_dir=mean_or_zero(e1["shift"]),
                         shift_opposite=mean_or_zero(e2["shift"]), rand_p95=p95, shift_over_p95=mean_or_zero(e1["shift"]) / max(p95, 0.05),
                         R_shift=mean_or_zero(e1["shift"]) / src["layer_eval"][L]["shift"]))
    return pd.DataFrame(rows).round(2)

tab = zh_shift_controls()
print("\n=== German order direction, margin shift by recipient (ZH is the new row) ===")
print(tab.to_string(index=False))

# ---- 4. dose-response for ZH (flip, random-direction flip, German source) ----
print("\n=== ZH dose-response ===")
print(pd.DataFrame(cell["dose"]).round(2).to_string(index=False))

z = tab[tab.recipient == "ZH"].iloc[0]
verdict = ("H_VO side: ZH moves like EN/FR/ES" if (z.R_shift >= 0.5 and z.shift_over_p95 >= 3 and z.shift_opposite < 0) else
           "H_dist side: ZH behaves like HI/JA/KO/TR" if (z.R_shift < 0.1 and z.shift_over_p95 < 3) else
           "INTERMEDIATE: do not interpret at n=12")
print(f"\nPRE-REGISTERED READ: {verdict}  (R_shift {z.R_shift}, shift {z.shift_dir} nats, {z.shift_over_p95}x random p95, opposite {z.shift_opposite})")
tab.to_csv(os.path.join(RESULTS_DIR, "exp29_order_shift_controls_with_ZH.csv"), index=False)
pd.DataFrame(cell["dose"]).to_csv(os.path.join(RESULTS_DIR, "exp29_dose_DE_to_ZH.csv"), index=False)

order ZH: acc=0.96  usable deriv/test=12/12  dropped(tokenisation)=0  -> PASS
  native margin (nats): mean 6.43, median 7.41, share of items with margin<1: 0.08

order DE->ZH: L=34  n=12  R(flip)=0.11 [0.00,0.36] ub=0.49  flip_rec=0.08333333333333333  R_shift=0.20  cue=0.08  ED50 ratio=0.07404203986064249  -> WALL 

=== German order direction, margin shift by recipient (ZH is the new row) ===
recipient  layer  n  clean_margin  shift_dir  shift_opposite  rand_p95  shift_over_p95  R_shift
       EN     36 12         11.72      10.42           -4.17      1.87            5.58     1.13
       FR     34 12         12.82       5.79           -3.95      1.20            4.84     0.63
       ES     36 12         11.34       6.96           -3.99      1.38            5.05     0.76
       HI     30 10          3.86       0.10           -0.64      0.21            0.48     0.01
       JA     30 11         13.77       0.76            0.17      0.28            2.73     0.08
       KO     30 12         

In [23]:
src = RESULTS["sources"][("order", "DE")]; a = src["alpha"]
rng = np.random.default_rng(1); rd = [unit_vec(rng.normal(size=hidden_dim())) for _ in range(30)]
for rec in ["FR", "ES", "HI", "AR"]:
    pT = RESULTS["ctxs"][("gender", rec)]["T"]
    for L in (30, 34, 36):
        sig = _sigma_at(pT, L)
        rs = [mean_or_zero(eff_steer(pT, L, d, a, sig)["shift"]) for d in rd]
        print(rec, L, "random p95 on gender margin:", round(float(np.quantile(rs, 0.95)), 2))

FR 30 random p95 on gender margin: 0.86
FR 34 random p95 on gender margin: 1.23
FR 36 random p95 on gender margin: 1.47
ES 30 random p95 on gender margin: 0.61
ES 34 random p95 on gender margin: 0.8
ES 36 random p95 on gender margin: 1.01
HI 30 random p95 on gender margin: 0.76
HI 34 random p95 on gender margin: 0.97
HI 36 random p95 on gender margin: 1.18
AR 30 random p95 on gender margin: 0.68
AR 34 random p95 on gender margin: 0.9
AR 36 random p95 on gender margin: 1.17
